# Metric Figures — FL Harmonization Article

Publication-grade metric and summary figures for the ComboBatch harmonization benchmark manuscript (Article 1).

**Cells 0–3:** Setup — imports, metrics data load, palettes, optional S3 raw expression load.  
**Cells 4+:** Figure cells — each saves SVG + PNG to `figures/` (main) or `figures/supplementary/` (supp).

---

## Part I — Setup

In [ ]:
from pathlib import Path
import json
import os
import boto3
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
import seaborn as sns
from scipy import stats
import matplotlib.patches as mpatches
import matplotlib.patches as patches
from scipy.stats import mannwhitneyu, pearsonr, spearmanr, median_abs_deviation
from statsmodels.stats.multitest import multipletests

plt.rcParams["pdf.fonttype"] = "truetype"
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["figure.dpi"] = 200
sns.set_style("ticks")

GLOBAL_FONT_SIZE = 7.5
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)
(FIGURES_DIR / "supplementary").mkdir(exist_ok=True)

BEST_LABEL = "Best"
BEST_COLOR = "#E63946"
BEST_HATCH = "///"

S3_BUCKET = os.environ.get("FL_S3_BUCKET", "")
METRICS_CSV_PATH = (
    Path("..") / "harmonization-metrics" / "metric_tables" / "metrics_comprehensive_260609.csv"
)

METRIC_TYPE_MAP = {
    "local_neighborhood": [
        "kbet_", "ilisi_", "clisi_",
        "graph_connectivity_", "umap_entropy_", "tsne_entropy_",
    ],
    "global_distance": [
        "r2_", "pcr_", "dsc_",
        "umap_centroid_disp_", "tsne_centroid_disp_",
        "dist_ratio_", "avg_intra_dist_", "avg_inter_dist_",
        "asw_batch_", "asw_bio_", "cms_",
        "pct_var_pc", "pct_var_cum", "wm_",
    ],
    "distribution_similarity": [
        "ks_mean_D_", "ks_frac_sig_",
        "ks_cohort_within_batch_", "per_gene_batch_mean_cv_",
    ],
    "other": ["pct_genes_", "pct_samples_", "pct_na_cells", "fraction_cohorts"],
}

GROUP_MAP = {
    "A": ["r2_", "pcr_", "dsc_"],
    "B": ["kbet_", "ilisi_", "clisi_", "asw_batch", "asw_bio", "cms_"],
    "C": ["umap_", "tsne_"],
    "D": ["ks_", "per_gene_batch"],
    "E": [
        "n_samples",
        "n_genes",
        "n_batches",
        "n_cohorts",
        "n_diagnosis_groups",
        "n_samples_per_batch",
        "zero_",
        "fraction_",
        "exp_",
        "per_batch_median",
        "bimodal",
        "below_1",
    ],
    "F": ["vp_"],
    "G": ["graph_"],
    "H": ["avg_intra_", "avg_inter_", "dist_ratio_"],
    "I": ["wm_"],
    "J": ["pct_var_pc", "pct_var_cum"],
    "K": [
        "n_genes_noNA",
        "pct_genes_noNA",
        "n_samples_noNA",
        "pct_samples_noNA",
        "n_genes_allNA",
        "n_samples_allNA",
        "n_na_cells",
        "pct_na_cells",
    ],
}
print("Cell 0 OK")

In [ ]:
import importlib, sys
sys.path.insert(0, str(Path(".").resolve()))
#import figures_helpers as fh
#importlib.reload(fh)
#save_figure = fh.save_figure

df_ok, df_normed, scoring_cols, col_meta = fh.load_metrics_data(METRICS_CSV_PATH)
scoring_cols=['asw_batch_norm_COHORT_LABEL',
 'asw_batch_norm_PLATFORM_RNA',
 'asw_batch_norm_RNASEQ_SOURCE',
 'asw_batch_norm_RNA_BATCH',
 'asw_bio_norm_Diagnosis_cell_type_unified',
 'asw_bio_norm_Major_group',
 'asw_bio_norm_TUMOR_NORMAL',
 'clisi_mean_Diagnosis_cell_type_unified',
 'clisi_mean_Major_group',
 'clisi_mean_TUMOR_NORMAL',
 'cms_fraction_mixed_COHORT_LABEL',
 'cms_fraction_mixed_PLATFORM_RNA',
 'cms_fraction_mixed_RNASEQ_SOURCE',
 'cms_fraction_mixed_RNA_BATCH',
 'dist_ratio_COHORT_LABEL',
 'dist_ratio_Diagnosis_cell_type_unified',
 'dist_ratio_Major_group',
 'dist_ratio_PLATFORM_RNA',
 'dist_ratio_RNASEQ_SOURCE',
 'dist_ratio_RNA_BATCH',
 'dist_ratio_TUMOR_NORMAL',
 'dsc_COHORT_LABEL',
 'dsc_PLATFORM_RNA',
 'dsc_RNASEQ_SOURCE',
 'dsc_RNA_BATCH',
 'fraction_cohorts_bimodal',
 'fraction_cohorts_zero_inflated_bimodal',
 'graph_connectivity_Diagnosis_cell_type_unified',
 'graph_connectivity_Major_group',
 'graph_connectivity_TUMOR_NORMAL',
 'ilisi_norm_COHORT_LABEL',
 'ilisi_norm_PLATFORM_RNA',
 'ilisi_norm_RNASEQ_SOURCE',
 'ilisi_norm_RNA_BATCH',
 'kbet_acceptance_rate_COHORT_LABEL',
 'kbet_acceptance_rate_PLATFORM_RNA',
 'kbet_acceptance_rate_RNASEQ_SOURCE',
 'kbet_acceptance_rate_RNA_BATCH',
 'ks_cohort_within_batch_frac_sig',
 'ks_cohort_within_batch_mean_D',
 'ks_frac_sig_COHORT_LABEL',
 'ks_frac_sig_PLATFORM_RNA',
 'ks_frac_sig_RNA_BATCH',
 'ks_mean_D_COHORT_LABEL',
 'ks_mean_D_PLATFORM_RNA',
 'ks_mean_D_RNA_BATCH',
 'pcr_COHORT_LABEL',
 'pcr_Diagnosis_cell_type_unified',
 'pcr_Major_group',
 'pcr_PLATFORM_RNA',
 'pcr_RNASEQ_SOURCE',
 'pcr_RNA_BATCH',
 'pcr_TUMOR_NORMAL',
 'pct_genes_noNA',
 'pct_na_cells',
 'pct_samples_noNA',
 'per_gene_batch_mean_cv_COHORT_LABEL',
 'per_gene_batch_mean_cv_PLATFORM_RNA',
 'per_gene_batch_mean_cv_RNA_BATCH',
 'tsne_centroid_disp_COHORT_LABEL',
 'tsne_centroid_disp_PLATFORM_RNA',
 'tsne_centroid_disp_RNASEQ_SOURCE',
 'tsne_centroid_disp_RNA_BATCH',
 'tsne_entropy_norm_COHORT_LABEL',
 'tsne_entropy_norm_PLATFORM_RNA',
 'tsne_entropy_norm_RNASEQ_SOURCE',
 'tsne_entropy_norm_RNA_BATCH',
 'umap_centroid_disp_COHORT_LABEL',
 'umap_centroid_disp_PLATFORM_RNA',
 'umap_centroid_disp_RNASEQ_SOURCE',
 'umap_centroid_disp_RNA_BATCH',
 'umap_entropy_norm_COHORT_LABEL',
 'umap_entropy_norm_PLATFORM_RNA',
 'umap_entropy_norm_RNASEQ_SOURCE',
 'umap_entropy_norm_RNA_BATCH',
 'wm_COHORT_LABEL',
 'wm_Diagnosis_cell_type_unified',
 'wm_Major_group',
 'wm_PLATFORM_RNA',
 'wm_RNASEQ_SOURCE',
 'wm_RNA_BATCH',
 'wm_TUMOR_NORMAL',
 'wm_mean_batch',
 'wm_mean_bio',
 'wm_ratio_bio_batch',
 'pct_samples_allNA',
 'pct_genes_allNA']
# 15 validated best approaches confirmed present in metrics_comprehensive_260609.csv.
# NOTE: the final article plan includes H_rnaseq_illumina and L_rnaseq_all_arrays
# strategies — update this list when those computations complete.
_top_ids = [
    ("10_mnn",      "strict",     "S0_no_removal"),
    ("10_mnn",      "strict",     "H_affymetrix_extended"),
    ("10_mnn",      "strict",     "D_malignant_only"),
    ("04_sva",      "knn",        "C_rnaseq_only"),
    ("04_sva",      "softimpute", "C_rnaseq_only"),
    ("16_fsqn_r",   "knn",        "C_rnaseq_only"),
    ("16_fsqn_r",   "softimpute", "C_rnaseq_only"),
    ("16_fsqn_r",   "strict",     "C_rnaseq_only"),
    ("10_mnn",      "strict",     "J_ff_only"),
    ("16_fsqn_r",   "strict",     "J_ff_only"),
    ("04_sva",      "knn",        "K_ffpe_only"),
    ("04_sva",      "strict",     "K_ffpe_only"),
    ("04_sva",      "softimpute", "K_ffpe_only"),
    ("13_fsmvn",    "strict",     "S0_no_removal"),
    ("33_amdbnorm", "strict",     "S0_no_removal"),
]

df_best, df_best_tagged, data_v3 = fh.build_best_vs_rest(df_ok, _top_ids, BEST_LABEL, BEST_COLOR)

assert len(df_best) == 15, f"Expected 15 best, got {len(df_best)}"
print(f"df_ok: {df_ok.shape} | df_best: {df_best.shape} | data_v3: {data_v3.shape}")
print(f"scoring_cols: {len(scoring_cols)} | col_meta groups: {col_meta['group'].unique()}")

In [ ]:
import matplotlib.colors as mcolors

# ── Strategy palette (fixed from v3 notebook) ────────────────────────────────
strat_pal = {
    "S0_no_removal":       (0.631, 0.788, 0.957),
    "A_confirmed_bad":     (1.000, 0.706, 0.510),
    "B_extended_bad":      (0.553, 0.898, 0.631),
    "C_rnaseq_only":       (1.000, 0.624, 0.608),
    "D_malignant_only":    (0.816, 0.733, 1.000),
    "E1_iterative_r1":     (0.871, 0.733, 0.608),
    "E2_iterative_r2":     (0.980, 0.690, 0.894),
    "E3_iterative_r3":     (0.812, 0.812, 0.812),
    "F_microarray_only":   (1.000, 0.996, 0.639),
    "G_affymetrix_only":   (0.725, 0.949, 0.941),
    "H_affymetrix_extended": (0.741, 0.718, 0.420),
    "I_rare_batches_removed": "#b095c5",
    "J_ff_only":           "#7ecbc4",
    "K_ffpe_only":         "#f2c27f",
    "H_rnaseq_illumina":   "#5fa8d3",
    "L_rnaseq_all_arrays":  "#e07b54",
}

# ── Method palette (tab20 + tab20b boosted saturation) ──────────────────────
_all_methods = sorted(df_ok["method"].unique())
_tab20 = sns.color_palette("tab20", n_colors=20)
_tab20b = sns.color_palette("tab20b", n_colors=20)
_raw_colors = (_tab20 + _tab20b)[:max(len(_all_methods), 25)]

def _boost_sat(rgb, delta=0.15):
    h, s, v = mcolors.rgb_to_hsv(np.clip(rgb[:3], 0, 1))
    return tuple(mcolors.hsv_to_rgb([h, min(1.0, s + delta), v]))

_vivid = [_boost_sat(c) for c in _raw_colors]
method_pal = dict(zip(_all_methods, _vivid[:len(_all_methods)]))
method_pal[BEST_LABEL] = BEST_COLOR

# ── Other palettes ───────────────────────────────────────────────────────────
imp_pal = {"strict": "#2979ae", "knn": "#982d22", "softimpute": "#713689"}
harshness_pal = {"low": "#2DC653", "medium": "#F4A261", "high": "#E63946"}
post_rm_pal = {False: "#CCCCCC", True: "#333333"}
group_pal = dict(zip("ABCDEFGHIJK", sns.color_palette("Set3", n_colors=11)))

# ── Harshness tier assignment ────────────────────────────────────────────────
HARSHNESS_MAP = {
    "S0_no_removal": "low", "A_confirmed_bad": "low", "B_extended_bad": "low",
    "C_rnaseq_only": "medium", "D_malignant_only": "medium",
    "E1_iterative_r1": "medium", "E2_iterative_r2": "medium", "E3_iterative_r3": "medium",
    "F_microarray_only": "medium", "G_affymetrix_only": "medium",
    "H_affymetrix_extended": "medium", "H_rnaseq_illumina": "medium",
    "I_rare_batches_removed": "low",
    "J_ff_only": "high", "K_ffpe_only": "high", "L_rnaseq_all_arrays": "medium",
}
df_ok["harshness"] = df_ok["strat"].map(HARSHNESS_MAP).fillna("medium")
data_v3["harshness"] = data_v3["strat"].map(HARSHNESS_MAP).fillna("medium")

metric_large_type_pal = {
    "local_neighborhood": "#1a6b3c",
    "global_distance": "#8b4a0f",
    "distribution_similarity": "#0d5c63",
    "other": "#5c5c1a",
}
HARSHNESS_LEVEL_MAP: dict[str, str] = {
    "01_raw": "low",
    "02_median_scaling": "low",
    "03_limma": "low",
    "04_sva": "low",
    "05_combat": "medium",
    "06_combat_seq": "medium",
    "07_pycombat": "medium",
    "08_inmoose_combatseq": "medium",
    "09_ruv": "medium",
    "10_mnn": "medium",
    "11_harmony": "medium",
    "12_scanorama": "medium",
    "13_fsmvn": "medium",
    "14_qsmooth": "high",
    "15_fsqn_py": "high",
    "16_fsqn_r": "high",
    "17_quantile": "high",
    "18_rank": "high",
    "19_tdm": "high",
    "20_shambhala": "high",
    "21_harmonizr": "medium",
    "22_tmm": "low",
    "23_vst": "low",
    "24_peer_k10": "medium",
    "25_angel": "high",
    "26_xpn": "high",
    "27_dwd": "medium",
    "28_npn": "high",
    "29_combat_ref": "medium",
    "30_recombat": "medium",
    "31_ruv3prps": "medium",
    "32_deepmnn": "medium",
    "33_amdbnorm": "high",
    "34_arsyn": "medium",
    "35_dasc": "medium",
    "36_explobatch": "medium",
    "37_fabatch": "medium",
    "38_harman": "medium",
    "39_procrustes": "high",
}
df_ok["harshness_level"] = df_ok["method"].apply(
    lambda m: HARSHNESS_LEVEL_MAP.get(
        m,
        "high" if m.startswith("shambhala_") else "medium",
    )
)

data_v3["harshness_level"] = data_v3["method"].apply(
    lambda m: HARSHNESS_LEVEL_MAP.get(
        m,
        "high" if m.startswith("shambhala_") else "medium",
    )
)

HARSHNESS_PAL = {
    "low": "#4caf50",
    "medium": "#ff9800",
    "high": "#f44336",
}

method_harshness_pal = (
    df_ok[["method", "harshness_level"]]
    .reset_index(drop=True)
    .drop_duplicates()
    .set_index("method")["harshness_level"]
    .map(HARSHNESS_PAL)
    .to_dict()
)

group_pal = dict(zip("ABCDEFGHIJK", sns.color_palette("Set3", n_colors=11)))


# ── Metric-type column lookup helper ────────────────────────────────────────
def get_metric_cols(metric_type: str, available_cols: list) -> list:
    """Return columns from available_cols matching any prefix in METRIC_TYPE_MAP[metric_type]."""
    prefixes = METRIC_TYPE_MAP.get(metric_type, [])
    return [c for c in available_cols if any(c.startswith(p) for p in prefixes)]

BATCH_ANNOT_COLS = ["RNA_BATCH", "PLATFORM_RNA", "RNASEQ_SOURCE", "COHORT_LABEL"]
BIO_ANNOT_COLS = ["Major_group", "Diagnosis_cell_type_unified", "TUMOR_NORMAL"]
GLOBAL_ANNOT_COL = "global/all"
_batch_pal_colors = ["#8B4513", "#CD853F", "#DEB887", "#F5DEB3"]
_bio_pal_colors = ["#2F4F4F", "#5F9EA0", "#B0C4DE"]
annot_pal = (
    dict(zip(BATCH_ANNOT_COLS, _batch_pal_colors))
    | dict(zip(BIO_ANNOT_COLS, _bio_pal_colors))
    | {GLOBAL_ANNOT_COL: "#808080"}
)


print("Cell 2 OK — palettes defined")
print(f"  methods: {len(method_pal)-1} + Best | strats: {len(strat_pal)}")
print(f"  harshness distribution: {df_ok['harshness'].value_counts().to_dict()}")

In [ ]:
# ── Cell 3: S3 data load — OPTIONAL ──────────────────────────────────────────
# Run this cell ONLY when producing Supp A, B, C (gene retention figures).
# Requires ~3 GB RAM. Skip for all other figures (Figs 7-11 need only df_ok).

s3 = boto3.client("s3")

print("Loading raw expression matrix from S3 (~2-3 min)...")
_obj = s3.get_object(
    Bucket=S3_BUCKET,
    Key="FL_batch_correction/exp/S0_no_removal__strict__01_raw__post0.tsv.gz",
)
comb_exp_raw = pd.read_csv(_obj["Body"], sep="\t", compression="gzip", index_col=0)
print(f"comb_exp_raw: {comb_exp_raw.shape}")

_obj = s3.get_object(
    Bucket=S3_BUCKET,
    Key="FL_batch_correction/prepared/S0_no_removal__knn__ann.tsv.gz",
)
comb_ann = pd.read_csv(_obj["Body"], sep="\t", compression="gzip", index_col=0)

# Copy batch_to_platform_group from Introductory_figures_for_article.ipynb cell 8
batch_to_platform_group: dict = {
    # RNA_BATCH_name: "Platform_group_string"
    # Populate from Introductory notebook
}
comb_ann["Platform_group"] = comb_ann["RNA_BATCH"].map(batch_to_platform_group)
print(f"comb_ann: {comb_ann.shape}")

# ── rna_batch_palette: used by Supp A for per-batch bar colors ────────────────
_rna_batches_sorted = sorted(comb_ann["RNA_BATCH"].unique())
_palette_colors = list(sns.color_palette("tab20", n_colors=min(20, len(_rna_batches_sorted))))
_palette_colors += list(
    sns.color_palette("tab20b", n_colors=max(0, len(_rna_batches_sorted) - 20))
)
rna_batch_palette = dict(zip(_rna_batches_sorted, _palette_colors))
print(f"rna_batch_palette: {len(rna_batch_palette)} batches")

In [ ]:
comb_ann

In [ ]:
print((~comb_ann.isna()).sum().sort_values()[::-1].to_string())

---

## Part II — Main and Supplementary Figures

### Figure 7 — Local Neighborhood Metrics

Four-panel overview of batch mixing quality measured at the local level:
- **kBET acceptance rate** — probability that a local neighborhood matches the global batch distribution (higher = better mixing)
- **iLISI × cLISI scatter** — jointly assesses batch integration (iLISI) and biology preservation (cLISI); ideal runs cluster top-right
- **UMAP / tSNE density entropy** — uniformity of batch labels in embedding neighborhoods
- **Local vs. global R²** — how much batch variance remains at the local vs. global scale after harmonization

In [ ]:
## Figure 7 — Local Neighborhood Metrics [MAIN or SUPPLEMENTARY — inspect output]

_kbet_col = "kbet_acceptance_rate_RNA_BATCH"
_ilisi_col = "ilisi_mean_RNA_BATCH"
_clisi_col = "clisi_mean_Diagnosis_cell_type_unified"
_pcr_col = "pcr_RNA_BATCH"

# Group G: graph connectivity availability check
_gc_cols = [c for c in df_ok.columns if c.startswith("graph_connectivity_")]
_gc_ok = bool(_gc_cols) and df_ok[_gc_cols].notna().mean().mean() > 0.5
print(f"Group G cols: {_gc_cols} | coverage: {df_ok[_gc_cols].notna().mean().mean():.1%}" if _gc_cols else "Group G: absent")

_umap_ent_cols = [c for c in scoring_cols if c.startswith("umap_entropy_")]
_tsne_ent_cols = [c for c in scoring_cols if c.startswith("tsne_entropy_")]
print(f"UMAP entropy cols: {_umap_ent_cols}")
print(f"tSNE entropy cols: {_tsne_ent_cols}")

fig = plt.figure(figsize=(8.27, 11.69))
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.60, wspace=0.42)
ax_A = fig.add_subplot(gs[0, 0])   # Panel A — kBET acceptance rate by method (strip, symlog)
ax_B = fig.add_subplot(gs[0, 1])   # Panel B — iLISI × cLISI trade-off scatter
ax_C = fig.add_subplot(gs[1, 0])   # Panel C — Graph connectivity by biology group (bar)
ax_D = fig.add_subplot(gs[1, 1])   # Panel D — tSNE entropy by method (boxplot)
ax_E = fig.add_subplot(gs[2, :])   # Panel E — Local (kBET) vs global (PCR) scatter (wide)

# ── Panel A: kBET acceptance rate by method — horizontal strip (symlog) ───────
_order_A = (
    df_ok.groupby("method")[_kbet_col].mean()
    .sort_values(ascending=False).index.tolist()
)
sns.stripplot(
    data=df_ok, y="method", x=_kbet_col, hue="method",
    order=_order_A, palette=method_pal, ax=ax_A,
    size=2, alpha=0.45, jitter=True, legend=False,
)
ax_A.axvline(
    df_best[_kbet_col].mean(), color=BEST_COLOR, linestyle="--", linewidth=1.2,
    label=f"Best mean: {df_best[_kbet_col].mean():.2f}",
)
ax_A.legend(fontsize=GLOBAL_FONT_SIZE - 2, loc="lower right")
ax_A.set_xscale("log")
ax_A.set_xlabel("kBET acceptance rate (log scale)", fontsize=GLOBAL_FONT_SIZE)
ax_A.set_ylabel("", fontsize=GLOBAL_FONT_SIZE)
ax_A.set_title("A: kBET by method (sorted, log)", fontsize=GLOBAL_FONT_SIZE)
ax_A.tick_params(labelsize=GLOBAL_FONT_SIZE - 2)
sns.despine(ax=ax_A)

# ── Panel B: iLISI (batch mixing) × cLISI (biology) trade-off scatter ────────
if _ilisi_col in df_ok.columns and _clisi_col in df_ok.columns:
    _samp_b = df_ok.sample(min(600, len(df_ok)), random_state=42)
    sns.scatterplot(
        data=_samp_b, x=_ilisi_col, y=_clisi_col, hue="method",
        palette=method_pal, ax=ax_B, s=18, alpha=0.5, legend=False,
    )
    ax_B.scatter(
        df_best[_ilisi_col], df_best[_clisi_col],
        marker="*", s=120, color=BEST_COLOR, zorder=5, label="Best ★",
    )
    ax_B.legend(fontsize=GLOBAL_FONT_SIZE - 2)
    ax_B.set_xlabel("iLISI RNA_BATCH\n(higher = better batch mixing)", fontsize=GLOBAL_FONT_SIZE)
    ax_B.set_ylabel("cLISI Diagnosis\n(higher = better bio preservation)", fontsize=GLOBAL_FONT_SIZE)
    ax_B.set_title("B: Batch–biology local mixing trade-off", fontsize=GLOBAL_FONT_SIZE)
    sns.despine(ax=ax_B)
else:
    ax_B.text(0.5, 0.5, f"iLISI/cLISI columns absent\n({_ilisi_col})",
              ha="center", va="center", transform=ax_B.transAxes, fontsize=GLOBAL_FONT_SIZE)

# ── Panel C: Graph connectivity by biology covariate — grouped barplot ────────
if _gc_ok:
    _bio_gc_cols = [c for c in _gc_cols if any(x in c for x in ("Diagnosis", "Major_group", "TUMOR_NORMAL"))]
    _gc_melted = df_ok.melt(
        id_vars=["method"], value_vars=_bio_gc_cols,
        var_name="covariate", value_name="gc",
    )
    _gc_melted["covariate"] = _gc_melted["covariate"].str.replace("graph_connectivity_", "", regex=False)
    _order_C = (
        df_ok.groupby("method")[_bio_gc_cols[0]].mean()
        .sort_values(ascending=False).index.tolist()
    )
    sns.barplot(
        data=_gc_melted, x="method", y="gc", hue="covariate",
        order=_order_C, ax=ax_C, palette="Set2",
    )
    ax_C.set_xticklabels(ax_C.get_xticklabels(), rotation=45, ha="right", fontsize=GLOBAL_FONT_SIZE - 2)
    ax_C.set_title("C: Graph connectivity by biology group", fontsize=GLOBAL_FONT_SIZE)
    ax_C.set_xlabel("", fontsize=GLOBAL_FONT_SIZE)
    ax_C.set_ylabel("Graph connectivity", fontsize=GLOBAL_FONT_SIZE)
    ax_C.legend(title="Covariate", fontsize=GLOBAL_FONT_SIZE - 2, loc="lower right")
    sns.despine(ax=ax_C)
else:
    ax_C.text(0.5, 0.5, "Group G: no data\n(graph_connectivity_* absent or < 50% coverage)",
              ha="center", va="center", transform=ax_C.transAxes, fontsize=GLOBAL_FONT_SIZE)

# ── Panel D: tSNE entropy (normalized) by method — boxplot ───────────────────
# UMAP entropy moved to Supplementary Figure (see next cell)
if _tsne_ent_cols:
    _ent_data_d = df_ok[["method"] + _tsne_ent_cols].melt(
        id_vars="method", var_name="covariate", value_name="entropy",
    )
    _order_D = (
        df_ok.groupby("method")[_tsne_ent_cols[0]].mean()
        .sort_values(ascending=False).index.tolist()
    )
    sns.boxplot(
        data=_ent_data_d, x="method", y="entropy", order=_order_D,
        palette=method_pal, ax=ax_D,
        flierprops={"marker": ".", "markersize": 2},
    )
    ax_D.set_xticklabels(ax_D.get_xticklabels(), rotation=45, ha="right", fontsize=GLOBAL_FONT_SIZE - 2)
    ax_D.set_title("D: tSNE entropy (norm) by method", fontsize=GLOBAL_FONT_SIZE)
    ax_D.set_xlabel("", fontsize=GLOBAL_FONT_SIZE)
    ax_D.set_ylabel("tSNE mixing entropy (normalized)", fontsize=GLOBAL_FONT_SIZE)
    sns.despine(ax=ax_D)
else:
    ax_D.text(0.5, 0.5, "tsne_entropy_* columns absent",
              ha="center", va="center", transform=ax_D.transAxes, fontsize=GLOBAL_FONT_SIZE)

# ── Panel E: Local (kBET) vs global (PCR) — key insight scatter (wide) ────────
if _kbet_col in df_ok.columns and _pcr_col in df_ok.columns:
    _samp_e = df_ok.sample(min(800, len(df_ok)), random_state=7)
    sns.scatterplot(
        data=_samp_e, x=_pcr_col, y=_kbet_col, hue="method",
        palette=method_pal, ax=ax_E, s=18, alpha=0.5, legend=False,
    )
    ax_E.scatter(
        df_best[_pcr_col], df_best[_kbet_col],
        marker="*", s=150, color=BEST_COLOR, zorder=5, label="Best ★",
    )
    ax_E.legend(fontsize=GLOBAL_FONT_SIZE - 2)
    ax_E.set_xlabel("PCR RNA_BATCH (lower = less batch variance)", fontsize=GLOBAL_FONT_SIZE)
    ax_E.set_ylabel("kBET acceptance rate (higher = better)", fontsize=GLOBAL_FONT_SIZE)
    ax_E.set_title(
        "E: Local–global metric correlation\n"
        "(kBET vs PCR — each point is one harmonization attempt)",
        fontsize=GLOBAL_FONT_SIZE,
    )
    sns.despine(ax=ax_E)
else:
    ax_E.text(0.5, 0.5, "kBET or PCR column absent",
              ha="center", va="center", transform=ax_E.transAxes, fontsize=GLOBAL_FONT_SIZE)

for ax in [ax_A, ax_B, ax_C, ax_D, ax_E]:
    ax.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
fig.suptitle("Figure 7 — Local Neighborhood Metrics", fontsize=GLOBAL_FONT_SIZE + 1, y=1.01)
plt.tight_layout()
plt.show()
save_figure(fig, "fig7_local_metrics_neighborhood")
plt.close()
print("Fig 7 saved.")

In [ ]:
## Supplementary Figure 7S — UMAP Entropy by Method (moved from Fig 7 Panel D)
# UMAP entropy was removed from the main Fig 7 to allow a wider local-global scatter.

if _umap_ent_cols:
    fig_umap_ent, ax_umap = plt.subplots(1, 1, figsize=(8.27, 5.0))
    _ent_data_umap = df_ok[["method"] + _umap_ent_cols].melt(
        id_vars="method", var_name="covariate", value_name="entropy",
    )
    _order_umap = (
        df_ok.groupby("method")[_umap_ent_cols[0]].mean()
        .sort_values(ascending=False).index.tolist()
    )
    sns.boxplot(
        data=_ent_data_umap, x="method", y="entropy", order=_order_umap,
        palette=method_pal, ax=ax_umap,
        flierprops={"marker": ".", "markersize": 2},
    )
    ax_umap.set_xticklabels(ax_umap.get_xticklabels(), rotation=45, ha="right",
                            fontsize=GLOBAL_FONT_SIZE - 2)
    ax_umap.set_title("Supp 7S: UMAP mixing entropy (norm) by method", fontsize=GLOBAL_FONT_SIZE)
    ax_umap.set_xlabel("", fontsize=GLOBAL_FONT_SIZE)
    ax_umap.set_ylabel("UMAP mixing entropy (normalized)", fontsize=GLOBAL_FONT_SIZE)
    ax_umap.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
    sns.despine(ax=ax_umap)
    fig_umap_ent.suptitle(
        "Supp 7S — UMAP Entropy by Method", fontsize=GLOBAL_FONT_SIZE + 1, y=1.02,
    )
    plt.tight_layout()
    save_figure(fig_umap_ent, "supplementary/supp7s_umap_entropy")
    plt.show()
    plt.close()
    print("Supp 7S saved.")
else:
    print("Supp 7S skipped — umap_entropy_* columns absent")

### Figure 8 — Structural and Distance Metrics

Measures whether harmonization collapses within-batch distances while preserving between-group biological distances:
- **Centroid dispersion heatmap** — per-batch centroid spread in PCA space, grouped by method
- **WassersteinMean / CMS / ASW** — global distributional and cluster separation metrics
- Rows: strategies; columns: methods; color: normalized metric score

In [ ]:
## Figure 8 — Structural and Distance Metrics [MAIN or SUPPLEMENTARY — inspect output]

# Group I (WaterMelon) availability
_wm_batch_col = "wm_mean_batch" if "wm_mean_batch" in df_ok.columns else None
_wm_bio_col = "wm_mean_bio" if "wm_mean_bio" in df_ok.columns else None
_wm_ratio_col = "wm_ratio_bio_batch" if "wm_ratio_bio_batch" in df_ok.columns else None
_wm_ok = bool(_wm_batch_col) and df_ok[[_wm_batch_col, _wm_bio_col]].notna().mean().mean() > 0.5
print(f"WaterMelon (Group I): {_wm_ok} | coverage: {df_ok[[_wm_batch_col, _wm_bio_col]].notna().mean().mean():.1%}" if _wm_batch_col else "WaterMelon: absent")

_disp_covariates = ["RNA_BATCH", "COHORT_LABEL", "PLATFORM_RNA", "RNASEQ_SOURCE"]
_disp_cols = [
    c for c in df_ok.columns
    if "centroid_disp" in c
    and any(cov in c for cov in _disp_covariates)
    and df_ok[c].notna().mean() > 0.5
]
_cms_cols = [c for c in scoring_cols if c.startswith("cms_")]
_asw_batch_cols = [c for c in scoring_cols if c.startswith("asw_batch_")]
_asw_bio_cols = [c for c in scoring_cols if c.startswith("asw_bio_")]
print(f"Centroid disp cols: {len(_disp_cols)} | CMS cols: {len(_cms_cols)} | ASW: {len(_asw_batch_cols)}+{len(_asw_bio_cols)}")

fig = plt.figure(figsize=(8.27, 11.69))
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.55, wspace=0.42)
ax_A = fig.add_subplot(gs[0, :])   # Panel A — centroid dispersion heatmap (wide)
ax_B = fig.add_subplot(gs[1, 0])   # Panel B — WaterMelon scatter (batch vs bio)
ax_C = fig.add_subplot(gs[1, 1])   # Panel C — WaterMelon ratio ranked barplot
ax_D = fig.add_subplot(gs[2, 0])   # Panel D — CMS by method (grouped barplot)
ax_E = fig.add_subplot(gs[2, 1])   # Panel E — ASW batch + ASW bio comparison

# ── Panel A: Centroid dispersion heatmap — methods × embedding×covariate ──────
if _disp_cols:
    _order_A = (
        df_ok.groupby("method")[_disp_cols].mean().mean(axis=1)
        .sort_values(ascending=True).index.tolist()
    )
    _disp_pivot = df_ok.groupby("method")[_disp_cols].mean().loc[_order_A]
    # Shorten column labels for readability
    _disp_labels = [c.replace("umap_centroid_disp_", "UMAP/").replace("tsne_centroid_disp_", "tSNE/") for c in _disp_pivot.columns]
    _disp_pivot.columns = _disp_labels
    sns.heatmap(
        _disp_pivot.T, ax=ax_A, cmap="RdBu_r", center=_disp_pivot.values.mean(),
        cbar_kws={"shrink": 0.5, "label": "Mean centroid dispersion"},
        xticklabels=True, yticklabels=True,
    )
    ax_A.set_title("A: UMAP/tSNE centroid dispersion by method (lower = better batch correction)",
                   fontsize=GLOBAL_FONT_SIZE)
    ax_A.tick_params(labelsize=GLOBAL_FONT_SIZE - 2)
    ax_A.set_xlabel("Method (sorted by mean dispersion)", fontsize=GLOBAL_FONT_SIZE)
else:
    ax_A.text(0.5, 0.5, "centroid_disp_* columns absent",
              ha="center", va="center", transform=ax_A.transAxes, fontsize=GLOBAL_FONT_SIZE)

# ── Panel B: WaterMelon batch vs bio scatter ──────────────────────────────────
if _wm_ok:
    _samp_b = df_ok.sample(min(600, len(df_ok)), random_state=17)
    sns.scatterplot(
        data=_samp_b, x=_wm_batch_col, y=_wm_bio_col, hue="method",
        palette=method_pal, ax=ax_B, s=18, alpha=0.5, legend=False,
    )
    ax_B.scatter(
        df_best[_wm_batch_col], df_best[_wm_bio_col],
        marker="*", s=120, color=BEST_COLOR, zorder=5, label="Best ★",
    )
    _wm_max = max(df_ok[_wm_batch_col].max(), df_ok[_wm_bio_col].max())
    _wm_min = min(df_ok[_wm_batch_col].min(), df_ok[_wm_bio_col].min())
    ax_B.plot([_wm_min, _wm_max], [_wm_min, _wm_max], "k--", linewidth=0.8, alpha=0.5, label="bio=batch")
    ax_B.legend(fontsize=GLOBAL_FONT_SIZE - 2)
    ax_B.set_xlabel("WaterMelon batch score (lower = better)", fontsize=GLOBAL_FONT_SIZE)
    ax_B.set_ylabel("WaterMelon biology score (higher = better)", fontsize=GLOBAL_FONT_SIZE)
    ax_B.set_title("B: WaterMelon batch vs. biology score", fontsize=GLOBAL_FONT_SIZE)
    sns.despine(ax=ax_B)
else:
    ax_B.text(0.5, 0.5, "WaterMelon (Group I): absent or sparse",
              ha="center", va="center", transform=ax_B.transAxes, fontsize=GLOBAL_FONT_SIZE)

# ── Panel C: WaterMelon ratio bio/batch — horizontal barplot ─────────────────
if _wm_ratio_col and df_ok[_wm_ratio_col].notna().mean() > 0.5:
    _wm_mean = df_ok.groupby("method")[_wm_ratio_col].mean().sort_values(ascending=True)
    _colors_C = [
        BEST_COLOR if m in df_best["method"].values else method_pal.get(m, "gray")
        for m in _wm_mean.index
    ]
    ax_C.barh(_wm_mean.index, _wm_mean.values, color=_colors_C, height=0.7)
    ax_C.axvline(1.0, linestyle="--", color="black", linewidth=0.8, label="ratio = 1")
    ax_C.set_xscale("symlog", linthresh=0.1)
    ax_C.legend(fontsize=GLOBAL_FONT_SIZE - 2)
    ax_C.set_xlabel("WM ratio (bio / batch)", fontsize=GLOBAL_FONT_SIZE)
    ax_C.set_title("C: WaterMelon bio/batch ratio by method\n(> 1 = biology > batch effect)", fontsize=GLOBAL_FONT_SIZE)
    ax_C.tick_params(labelsize=GLOBAL_FONT_SIZE - 2)
    sns.despine(ax=ax_C)
else:
    ax_C.text(0.5, 0.5, "wm_ratio_bio_batch absent or sparse",
              ha="center", va="center", transform=ax_C.transAxes, fontsize=GLOBAL_FONT_SIZE)

# ── Panel D: CMS (fraction mixed) by method — grouped barplot ────────────────
if _cms_cols:
    _cms_melted = df_ok.melt(
        id_vars=["method"], value_vars=_cms_cols,
        var_name="covariate", value_name="cms",
    )
    _cms_melted["covariate"] = _cms_melted["covariate"].str.replace("cms_fraction_mixed_", "", regex=False)
    _order_D = df_ok.groupby("method")[_cms_cols[0]].mean().sort_values(ascending=False).index.tolist()
    sns.barplot(
        data=_cms_melted, x="method", y="cms", hue="covariate",
        order=_order_D, ax=ax_D, palette="Paired",
    )
    ax_D.set_xticklabels(ax_D.get_xticklabels(), rotation=45, ha="right", fontsize=GLOBAL_FONT_SIZE - 2)
    ax_D.set_title("D: CMS fraction mixed by method\n(higher = better batch mixing)", fontsize=GLOBAL_FONT_SIZE)
    ax_D.set_xlabel("", fontsize=GLOBAL_FONT_SIZE)
    ax_D.set_ylabel("CMS fraction mixed", fontsize=GLOBAL_FONT_SIZE)
    ax_D.legend(title="Covariate", fontsize=GLOBAL_FONT_SIZE - 2)
    sns.despine(ax=ax_D)
else:
    ax_D.text(0.5, 0.5, "CMS columns absent",
              ha="center", va="center", transform=ax_D.transAxes, fontsize=GLOBAL_FONT_SIZE)

# ── Panel E: ASW batch + ASW bio by method — grouped barplot ─────────────────
if _asw_batch_cols and _asw_bio_cols:
    # Use first (RNA_BATCH / Diagnosis) representative column per type
    _asw_batch_rep = next((c for c in _asw_batch_cols if "RNA_BATCH" in c), _asw_batch_cols[0])
    _asw_bio_rep = next((c for c in _asw_bio_cols if "Diagnosis" in c), _asw_bio_cols[0])
    _asw_df = pd.DataFrame({
        "method": df_ok["method"],
        "asw_batch_norm": df_ok[_asw_batch_rep],
        "asw_bio_norm": df_ok[_asw_bio_rep],
    })
    _asw_mean = _asw_df.groupby("method")[["asw_batch_norm", "asw_bio_norm"]].mean()
    _order_E = _asw_mean["asw_batch_norm"].sort_values().index.tolist()
    _asw_melted = _asw_mean.loc[_order_E].reset_index().melt(
        id_vars="method", var_name="metric", value_name="score",
    )
    sns.barplot(
        data=_asw_melted, x="method", y="score", hue="metric",
        order=_order_E, ax=ax_E,
        palette={"asw_batch_norm": "#E76F51", "asw_bio_norm": "#2A9D8F"},
    )
    ax_E.set_xticklabels(ax_E.get_xticklabels(), rotation=45, ha="right", fontsize=GLOBAL_FONT_SIZE - 2)
    ax_E.set_title("E: ASW batch vs. biology (silhouette width norm.)", fontsize=GLOBAL_FONT_SIZE)
    ax_E.set_xlabel("", fontsize=GLOBAL_FONT_SIZE)
    ax_E.set_ylabel("ASW normalized", fontsize=GLOBAL_FONT_SIZE)
    ax_E.legend(title="Metric", fontsize=GLOBAL_FONT_SIZE - 2, labels=["ASW batch (↑ = less batch)", "ASW biology (↑ = more bio)"])
    sns.despine(ax=ax_E)
else:
    ax_E.text(0.5, 0.5, "ASW batch/bio columns absent",
              ha="center", va="center", transform=ax_E.transAxes, fontsize=GLOBAL_FONT_SIZE)

for ax in [ax_B, ax_C, ax_D, ax_E]:
    ax.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
fig.suptitle("Figure 8 — Structural and Distance Metrics", fontsize=GLOBAL_FONT_SIZE + 1, y=1.01)
plt.tight_layout()
plt.show()
save_figure(fig, "fig8_structural_distance_metrics")
plt.close()
print("Fig 8 saved.")

### Figure 8B — Distributional Similarity and NA Retention

How closely each method reproduces the reference expression distribution, and how much data is lost:
- **KS-statistic** — gene-wise Kolmogorov–Smirnov distance between harmonized and reference distributions (lower = closer)
- **Coefficient of variation** — stability of expression variance across samples post-harmonization
- **NA gene heatmap by batch** — which batches still carry missing values after each imputation strategy
- **Non-NA gene % by harshness tier** — data retention trade-off across strategy difficulty levels

In [ ]:
## Figure 8B — Distributional Similarity Metrics and NA Retention [MAIN or SUPPLEMENTARY]

_ks_d_cols = [c for c in scoring_cols if c.startswith("ks_mean_D_")]
_ks_sig_cols = [c for c in scoring_cols if c.startswith("ks_frac_sig_")]
_cv_cols = [c for c in scoring_cols if c.startswith("per_gene_batch_mean_cv_")]
_bimodal_cols = [c for c in scoring_cols if c.startswith("ks_cohort_within_batch_")]
print(f"KS cols — D: {_ks_d_cols} | sig: {_ks_sig_cols} | CV: {_cv_cols} | bimodal proxy: {_bimodal_cols}")

fig = plt.figure(figsize=(8.27, 11.69))
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.55, wspace=0.42)
ax_A = fig.add_subplot(gs[0, 0])   # Panel A — KS mean D by method (barplot)
ax_B = fig.add_subplot(gs[0, 1])   # Panel B — Fraction significant KS (stacked bar)
ax_C = fig.add_subplot(gs[1, 0])   # Panel C — Per-gene batch mean CV by method (boxplot)
ax_D = fig.add_subplot(gs[1, 1])   # Panel D — KS cohort-within-batch score by method
ax_E = fig.add_subplot(gs[2, 0])   # Panel E — NA cells heatmap (strategy × method)
ax_F = fig.add_subplot(gs[2, 1])   # Panel F — % noNA genes by harshness × imputation

# ── Panel A: KS mean D by method — horizontal barplot sorted ascending ────────
if _ks_d_cols:
    _ks_mean = data_v3.groupby("method")[_ks_d_cols[0]].mean().sort_values(ascending=True)
    _colors_A = [
        BEST_COLOR if m == BEST_LABEL else method_pal.get(m, "gray")
        for m in _ks_mean.index
    ]
    ax_A.barh(_ks_mean.index, _ks_mean.values, color=_colors_A, height=0.7)
    if BEST_LABEL in _ks_mean.index:
        ax_A.axvline(_ks_mean[BEST_LABEL], color=BEST_COLOR, linestyle="--",
                     linewidth=1.2, label=f"Best mean: {_ks_mean[BEST_LABEL]:.3f}")
        ax_A.legend(fontsize=GLOBAL_FONT_SIZE - 1)
    ax_A.set_xlabel(f"KS mean D — {_ks_d_cols[0].replace('ks_mean_D_','')}\n(lower = better)", fontsize=GLOBAL_FONT_SIZE)
    ax_A.set_title("A: KS test mean D by method (Best ★ highlighted)", fontsize=GLOBAL_FONT_SIZE)
    ax_A.tick_params(labelsize=GLOBAL_FONT_SIZE - 2)
    sns.despine(ax=ax_A)
else:
    ax_A.text(0.5, 0.5, "ks_mean_D_* columns absent",
              ha="center", va="center", transform=ax_A.transAxes, fontsize=GLOBAL_FONT_SIZE)

# ── Panel B: Fraction significant KS — stacked bar by method ─────────────────
if _ks_sig_cols:
    _ks_sig_mean = data_v3.groupby("method")[_ks_sig_cols].mean()
    _ks_sig_mean.columns = [c.replace("ks_frac_sig_", "") for c in _ks_sig_mean.columns]
    _ks_order_B = _ks_sig_mean.sum(axis=1).sort_values(ascending=True).index
    _ks_sig_mean.loc[_ks_order_B].plot(
        kind="barh", ax=ax_B, stacked=True, colormap="Set2", width=0.7,
    )
    ax_B.set_xlabel("Fraction significant KS tests", fontsize=GLOBAL_FONT_SIZE)
    ax_B.set_title("B: Fraction significant KS tests by covariate\n(lower = better)", fontsize=GLOBAL_FONT_SIZE)
    ax_B.legend(title="Covariate", fontsize=GLOBAL_FONT_SIZE - 2, loc="lower right")
    ax_B.tick_params(labelsize=GLOBAL_FONT_SIZE - 2)
    sns.despine(ax=ax_B)
else:
    ax_B.text(0.5, 0.5, "ks_frac_sig_* columns absent",
              ha="center", va="center", transform=ax_B.transAxes, fontsize=GLOBAL_FONT_SIZE)

# ── Panel C: Per-gene batch mean CV by method — horizontal boxplot ────────────
if _cv_cols:
    _cv_melted = df_ok[["method"] + _cv_cols].melt(
        id_vars="method", var_name="covariate", value_name="cv",
    )
    _order_C = df_ok.groupby("method")[_cv_cols[0]].mean().sort_values(ascending=True).index.tolist()
    sns.boxplot(
        data=_cv_melted, x="cv", y="method", order=_order_C,
        palette=method_pal, ax=ax_C,
        flierprops={"marker": ".", "markersize": 2},
    )
    ax_C.set_xlabel("Per-gene batch mean CV (lower = better)", fontsize=GLOBAL_FONT_SIZE)
    ax_C.set_ylabel("", fontsize=GLOBAL_FONT_SIZE)
    ax_C.set_title("C: Per-gene batch CV by method", fontsize=GLOBAL_FONT_SIZE)
    ax_C.tick_params(labelsize=GLOBAL_FONT_SIZE - 2)
    sns.despine(ax=ax_C)
else:
    ax_C.text(0.5, 0.5, "per_gene_batch_mean_cv_* columns absent",
              ha="center", va="center", transform=ax_C.transAxes, fontsize=GLOBAL_FONT_SIZE)

# ── Panel D: KS cohort-within-batch score (distributional bimodality proxy) ───
if _bimodal_cols:
    # Use mean_D column if available, else first available
    _bm_col = next((c for c in _bimodal_cols if "mean_D" in c), _bimodal_cols[0])
    _bm_mean = df_ok.groupby("method")[_bm_col].mean().sort_values(ascending=True)
    _colors_D = [
        BEST_COLOR if m in df_best["method"].values else method_pal.get(m, "gray")
        for m in _bm_mean.index
    ]
    ax_D.barh(_bm_mean.index, _bm_mean.values, color=_colors_D, height=0.7)
    ax_D.set_xlabel(f"{_bm_col.replace('ks_cohort_within_batch_','ks_')}\n(lower = better within-batch uniformity)", fontsize=GLOBAL_FONT_SIZE)
    ax_D.set_title("D: KS cohort-within-batch D by method", fontsize=GLOBAL_FONT_SIZE)
    ax_D.tick_params(labelsize=GLOBAL_FONT_SIZE - 2)
    sns.despine(ax=ax_D)
else:
    ax_D.text(0.5, 0.5, "ks_cohort_within_batch_* columns absent",
              ha="center", va="center", transform=ax_D.transAxes, fontsize=GLOBAL_FONT_SIZE)

# ── Panel E: NA cells heatmap — strategy × method ────────────────────────────
if "pct_na_cells" in df_ok.columns:
    _na_pivot = df_ok.groupby(["strat", "method"])["pct_na_cells"].mean().unstack()
    # Sort strategies by total NA and methods by total NA
    _na_pivot = _na_pivot.loc[
        _na_pivot.mean(axis=1).sort_values(ascending=False).index,
        _na_pivot.mean(axis=0).sort_values(ascending=False).index,
    ]
    sns.heatmap(
        _na_pivot, ax=ax_E, cmap="YlOrRd", annot=False,
        xticklabels=True, yticklabels=True,
        cbar_kws={"shrink": 0.5, "label": "% NA cells"},
    )
    ax_E.set_title("E: NA cells (%) — strategy × method", fontsize=GLOBAL_FONT_SIZE)
    ax_E.tick_params(labelsize=GLOBAL_FONT_SIZE - 2)
    ax_E.set_xlabel("Method", fontsize=GLOBAL_FONT_SIZE)
    ax_E.set_ylabel("Strategy", fontsize=GLOBAL_FONT_SIZE)
else:
    ax_E.text(0.5, 0.5, "pct_na_cells column absent",
              ha="center", va="center", transform=ax_E.transAxes, fontsize=GLOBAL_FONT_SIZE)

# ── Panel F: % noNA genes by harshness × imputation — grouped barplot ─────────
if "pct_genes_noNA" in df_ok.columns and "harshness" in df_ok.columns:
    _nona_data = df_ok.groupby(["imp", "harshness"])["pct_genes_noNA"].mean().reset_index()
    _harshness_order = ["low", "medium", "high"]
    sns.barplot(
        data=_nona_data, x="harshness", y="pct_genes_noNA", hue="imp",
        order=_harshness_order, palette=imp_pal, ax=ax_F,
    )
    ax_F.set_xlabel("Strategy harshness", fontsize=GLOBAL_FONT_SIZE)
    ax_F.set_ylabel("% non-NA genes retained", fontsize=GLOBAL_FONT_SIZE)
    ax_F.set_title("F: Gene coverage by harshness and imputation", fontsize=GLOBAL_FONT_SIZE)
    ax_F.legend(title="Imputation", fontsize=GLOBAL_FONT_SIZE - 1)
    sns.despine(ax=ax_F)
else:
    ax_F.text(0.5, 0.5, "pct_genes_noNA or harshness absent",
              ha="center", va="center", transform=ax_F.transAxes, fontsize=GLOBAL_FONT_SIZE)

for ax in [ax_A, ax_B, ax_C, ax_D, ax_E, ax_F]:
    ax.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
fig.suptitle("Figure 8B — Distributional Similarity and NA Retention", fontsize=GLOBAL_FONT_SIZE + 1, y=1.01)
plt.tight_layout()
plt.show()
save_figure(fig, "fig8b_distributional_similarity_na_retention")
plt.close()
print("Fig 8B saved.")

### Supplementary Figure 9 — PCA / UMAP / tSNE Embedding Comparison *(placeholder)*

3 × 3 placeholder grid reserved for low-dimensional embedding plots of the best harmonization approaches.  
Actual 2-D coordinates are produced by an external pipeline (Tolya); replace placeholder panels once coordinates are available.

In [ ]:
## Supplementary Figure 9 — PCA/UMAP/tSNE Embeddings with Best Approach Highlights
# PLACEHOLDER — requires pre-computed embedding coordinates from the visual inspection notebook
# or from S3 embedding outputs. Implement after confirming coordinate availability.
#
# Layout: 3 rows (PCA / UMAP / tSNE) × 3 cols (3 representative strategies)
# Best 15 highlighted with ★ markers in each panel.
# To implement: load embedding_coords_dict from visual inspection notebook outputs,
# then scatter each embedding panel with hue="method", palette=method_pal,
# overlay df_best points as stars.

fig = plt.figure(figsize=(8.27, 11.69))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

for row_i, emb_type in enumerate(["PCA", "UMAP", "tSNE"]):
    for col_i, strat_label in enumerate(["S0_no_removal", "C_rnaseq_only", "J_ff_only"]):
        ax = fig.add_subplot(gs[row_i, col_i])
        ax.text(
            0.5, 0.5,
            f"{emb_type}\n{strat_label}\n[embed coords not loaded]",
            ha="center", va="center", transform=ax.transAxes,
            fontsize=GLOBAL_FONT_SIZE, color="#888888",
        )
        ax.set_title(f"{emb_type} — {strat_label}", fontsize=GLOBAL_FONT_SIZE)
        ax.set_xticks([])
        ax.set_yticks([])
        sns.despine(ax=ax, left=True, bottom=True)

fig.suptitle("Supp Fig 9 — PCA/UMAP/tSNE Embeddings with Best ★ Highlights",
             fontsize=GLOBAL_FONT_SIZE + 1, y=1.01)
plt.tight_layout()
plt.show()
save_figure(fig, "supplementary/supp9_embeddings_best_stars")
plt.close()
print("Supp Fig 9 placeholder saved.")

### Figure 9 — Cross-Metric Correlation and Factor Importance

Examines which experimental factors drive metric outcomes and how global and local metrics relate to each other:
- **(A) PCR vs. kBET scatter** — colored by harshness tier; reveals the global–local trade-off: aggressive strategies improve PCR but reduce kBET
- **(B) Composite score violin** — distribution of equal-weight composite scores across harshness tiers
- **(C) η² effect sizes** — fraction of total metric variance explained by each factor (strategy > method > post-removal > imputation)
- **(D) Metric-type composition** — stacked bar showing which metric categories each method excels in

In [ ]:
len(scoring_cols)

In [ ]:
## Figure 9 — Cross-Metric Correlation and Factor Importance [MAIN]
# Key insight: strategy is the dominant factor (η²); local and global metrics disagree on SVA

from scipy import stats as scipy_stats

fig = plt.figure(figsize=(8.27, 11.69))
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.55, wspace=0.42)
ax_A = fig.add_subplot(gs[0, :])   # Panel A — local vs global scatter (wide), colored by harshness
ax_B = fig.add_subplot(gs[1, 0])   # Panel B — composite score by harshness tier (violin+strip)
ax_C = fig.add_subplot(gs[1, 1])   # Panel C — factor importance η² barplot
ax_D = fig.add_subplot(gs[2, :])   # Panel D — metric type score by method (stacked bar, wide)

# Compute composite score (equal weight) for Panel B
df_ok["composite_score"] = df_normed[scoring_cols].mean(axis=1)

# ── Panel A: PCR (global) vs kBET (local) — colored by harshness, Best ★ ─────
_repr_global = "pcr_RNA_BATCH"
_repr_local = next((c for c in df_ok.columns if c.startswith("kbet_acceptance_rate_")), None)
if _repr_global in df_ok.columns and _repr_local:
    _samp_a = df_ok.sample(min(800, len(df_ok)), random_state=99)
    sns.scatterplot(
        data=_samp_a, x=_repr_global, y=_repr_local,
        hue="harshness", palette=harshness_pal,
        ax=ax_A, s=22, alpha=0.65, legend="full",
    )
    ax_A.scatter(
        df_best[_repr_global], df_best[_repr_local],
        marker="*", s=180, color=BEST_COLOR, zorder=6, label="Best ★",
    )
    ax_A.legend(title="Harshness / Best", fontsize=GLOBAL_FONT_SIZE, ncol=2)
    ax_A.set_xlabel(
        f"PCR RNA_BATCH — global metric (lower = less batch effect)",
        fontsize=GLOBAL_FONT_SIZE,
    )
    ax_A.set_ylabel(
        f"kBET acceptance rate — local metric (higher = better mixing)",
        fontsize=GLOBAL_FONT_SIZE,
    )
    ax_A.set_title(
        f"A: Local vs. global batch correction metrics (n={len(df_ok):,}) — SVA achieves high kBET with moderate PCR",
        fontsize=GLOBAL_FONT_SIZE,
    )
    # Ring size encodes iLISI quartile as secondary local-mixing indicator
    _ilisi_col_a = next((c for c in df_ok.columns if c.startswith("ilisi_norm_RNA_BATCH")), None)
    if _ilisi_col_a and _ilisi_col_a in df_ok.columns:
        _ilisi_q = pd.qcut(df_ok[_ilisi_col_a].fillna(0), q=4,
                           labels=["q1", "q2", "q3", "q4"])
        _ilisi_sizes = _ilisi_q.map({"q1": 8, "q2": 15, "q3": 25, "q4": 40}).fillna(8)
        ax_A.scatter(df_ok[_repr_global], df_ok[_repr_local],
                     s=_ilisi_sizes, c="none", edgecolors="#333333",
                     linewidths=0.4, alpha=0.3, zorder=4,
                     label="ring size ∝ iLISI quartile")
    ax_A.legend(title="Harshness / Best", fontsize=GLOBAL_FONT_SIZE, ncol=2)
    sns.despine(ax=ax_A)
else:
    ax_A.text(0.5, 0.5, "PCR or kBET column absent", ha="center", va="center",
              transform=ax_A.transAxes, fontsize=GLOBAL_FONT_SIZE)

# ── Panel B: Composite score by harshness — violin + strip + Best ★ ──────────
sns.violinplot(
    data=df_ok, x="harshness", y="composite_score", palette=harshness_pal,
    order=["low", "medium", "high"], ax=ax_B, inner=None, alpha=0.55,
)
#sns.stripplot(data=df_ok, x="harshness", y="composite_score",order=["low", "medium", "high"], ax=ax_B,color="black", size=1.8, alpha=0.25, jitter=True,)
# Overlay Best ★
if df_best.index.isin(df_ok.index).all():
    _best_h = df_ok.loc[df_best.index, ["harshness", "composite_score"]]
    _harshness_jitter = {"low": -0.15, "medium": 0.0, "high": 0.15}
    _h_order = ["low", "medium", "high"]
    for h in _h_order:
        _sub = _best_h[_best_h["harshness"] == h]
        if len(_sub):
            ax_B.scatter(
                [_h_order.index(h)] * len(_sub), _sub["composite_score"],
                marker="*", s=100, color='black', zorder=6,
                facecolors="none",
                edgecolors='black',
                linewidths=0.7, alpha=0.85,
            )
ax_B.set_title("B: Composite score\nby strategy harshness", fontsize=GLOBAL_FONT_SIZE)
ax_B.set_xlabel("Harshness tier", fontsize=GLOBAL_FONT_SIZE)
ax_B.set_ylabel("Composite score (mean, 85 metrics)", fontsize=GLOBAL_FONT_SIZE)
ax_B.tick_params(labelsize=GLOBAL_FONT_SIZE)
sns.despine(ax=ax_B)

# ── Panel C: Factor importance η² — ANOVA across all 85 scoring_cols ──────────
# Cast post_rm to string so groupby works regardless of dtype
df_ok["post_rm_str"] = df_ok["post_rm"].astype(str)
factors = [("strat", "Strategy"), ("method", "Method"), ("post_rm_str", "Post-removal"), ("imp", "Imputation")]
eta2_results = {}
for factor_col, factor_label in factors:
    eta2_vals = []
    for col in scoring_cols:
        _sub = df_ok[[factor_col, col]].dropna()
        if _sub[factor_col].nunique() < 2 or _sub[col].std() == 0:
            continue
        groups = [g[col].values for _, g in _sub.groupby(factor_col)]
        _grand_mean = _sub[col].mean()
        ss_total = ((_sub[col] - _grand_mean) ** 2).sum()
        ss_between = sum(len(g) * (g.mean() - _grand_mean) ** 2 for g in groups)
        eta2_vals.append(ss_between / ss_total if ss_total > 0 else 0)
    eta2_results[factor_label] = float(np.mean(eta2_vals)) if eta2_vals else 0.0

_eta2_df = pd.Series(eta2_results).sort_values(ascending=True)
_eta2_colors = ["#2DC653", "#F4A261", "#E63946", "#457B9D"][::-1][:len(_eta2_df)]
ax_C.barh(_eta2_df.index, _eta2_df.values, color=_eta2_colors[::-1], height=0.6)
for i, (label, val) in enumerate(_eta2_df.items()):
    ax_C.text(val + 0.003, i, f"{val:.3f}", va="center", fontsize=GLOBAL_FONT_SIZE)
ax_C.set_xlabel("Mean η² effect size (across 85 scoring metrics)", fontsize=GLOBAL_FONT_SIZE)
ax_C.set_title("C: Factor importance: strategy > method > post-removal > imputation",
               fontsize=GLOBAL_FONT_SIZE)
ax_C.tick_params(labelsize=GLOBAL_FONT_SIZE)
sns.despine(ax=ax_C)

# ── Panel D: Mean normalized score by metric type and method — stacked bar ────
_type_scores = {}
for mtype, prefixes in METRIC_TYPE_MAP.items():
    _type_cols = [c for c in scoring_cols if any(c.startswith(p) for p in prefixes)]
    if _type_cols:
        _type_scores[mtype] = df_normed[_type_cols].mean(axis=1)
_type_df = pd.DataFrame(_type_scores)
_type_df["method"] = df_ok["method"].values
_type_mean = _type_df.groupby("method").mean()
_order_D = _type_mean.sum(axis=1).sort_values(ascending=False).index.tolist()
_type_mean.loc[_order_D].plot(
    kind="bar", stacked=True, ax=ax_D, colormap="Set2", width=0.8,
)
ax_D.set_xticklabels(ax_D.get_xticklabels(), rotation=45, ha="right",
                     fontsize=GLOBAL_FONT_SIZE)
ax_D.set_xlabel("Method (sorted by total composite score)", fontsize=GLOBAL_FONT_SIZE)
ax_D.set_ylabel(
    "Cumulative summed normalized score\n(stacked; not bounded to 1)",
    fontsize=GLOBAL_FONT_SIZE,
)
ax_D.set_title("D: Score by metric type and method (stacked — best methods upper right)",
               fontsize=GLOBAL_FONT_SIZE)
ax_D.legend(title="Metric type", fontsize=GLOBAL_FONT_SIZE, ncol=2, loc="upper right")
ax_D.tick_params(labelsize=GLOBAL_FONT_SIZE)
sns.despine(ax=ax_D)

fig.suptitle("Figure 9 — Cross-Metric Correlation and Factor Importance",
             fontsize=GLOBAL_FONT_SIZE, y=1.04)
plt.tight_layout()
plt.show()
save_figure(fig, "fig9_cross_metric_correlation_factor_importance")
plt.close()
print("Fig 9 saved.")
print(f"Factor importance ranking: {dict(_eta2_df.sort_values(ascending=False))}")

In [ ]:
df_ok["post_rm_str"] = df_ok["post_rm"].astype(str)
factors = [("strat", "Strategy"), ("method", "Method"), ("post_rm_str", "Post-removal"), ("imp", "Imputation")]
eta2_results = {}
for factor_col, factor_label in factors:
    eta2_vals = []
    for col in scoring_cols:
        _sub = df_ok[[factor_col, col]].dropna()
        if _sub[factor_col].nunique() < 2 or _sub[col].std() == 0:
            continue
        groups = [g[col].values for _, g in _sub.groupby(factor_col)]
        _grand_mean = _sub[col].mean()
        ss_total = ((_sub[col] - _grand_mean) ** 2).sum()
        ss_between = sum(len(g) * (g.mean() - _grand_mean) ** 2 for g in groups)
        eta2_vals.append(ss_between / ss_total if ss_total > 0 else 0)
    eta2_results[factor_label] = float(np.mean(eta2_vals)) if eta2_vals else 0.0

_eta2_df = pd.Series(eta2_results).sort_values(ascending=True)
_eta2_colors = ["#2DC653", "#F4A261", "#E63946", "#457B9D"][::-1][:len(_eta2_df)]

#### Confidense intervals

In [ ]:
df_ok["post_rm_str"] = df_ok["post_rm"].astype(str)
factors = [("strat", "Strategy"), ("method", "Method"), ("post_rm_str", "Post-removal"), ("imp", "Imputation")]

# Store dictionaries instead of a single float to capture CI bounds
eta2_results = []

for factor_col, factor_label in factors:
    eta2_vals = []
    for col in scoring_cols:
        _sub = df_ok[[factor_col, col]].dropna()
        
        if _sub[factor_col].nunique() < 2 or _sub[col].std() == 0:
            continue
            
        groups = [g[col].values for _, g in _sub.groupby(factor_col)]
        _grand_mean = _sub[col].mean()
        
        ss_total = ((_sub[col] - _grand_mean) ** 2).sum()
        ss_between = sum(len(g) * (g.mean() - _grand_mean) ** 2 for g in groups)
        
        eta2 = ss_between / ss_total if ss_total > 0 else 0
        eta2_vals.append(eta2)
        
    # Calculate Mean and 95% CI across the scoring columns
    if len(eta2_vals) > 1:
        mean_eta2 = float(np.mean(eta2_vals))
        se = stats.sem(eta2_vals)
        # Calculate CI using the t-distribution
        ci_low, ci_high = stats.t.interval(0.95, df=len(eta2_vals)-1, loc=mean_eta2, scale=se)
        
        # Bound the CIs so they don't fall outside the valid [0, 1] range for eta-squared
        ci_low = max(0.0, ci_low)
        ci_high = min(1.0, ci_high)
        
    elif len(eta2_vals) == 1:
        mean_eta2, ci_low, ci_high = float(eta2_vals[0]), float(eta2_vals[0]), float(eta2_vals[0])
    else:
        mean_eta2, ci_low, ci_high = 0.0, 0.0, 0.0

    eta2_results.append({
        "Factor": factor_label,
        "eta2_mean": mean_eta2,
        "ci_lower": ci_low,
        "ci_upper": ci_high
    })

# Convert to a DataFrame and sort by the mean
_eta2_df = pd.DataFrame(eta2_results).set_index("Factor").sort_values(by="eta2_mean", ascending=True)

# Generate colors matching the length of the resulting DataFrame
_eta2_colors = ["#2DC653", "#F4A261", "#E63946", "#457B9D"][::-1][:len(_eta2_df)]

In [ ]:
_eta2_df

#### Composite score statistical analysis

In [ ]:
df_ok[['composite_score', 'harshness']].groupby('harshness').median()

In [ ]:
df_best = df_ok.reindex(df_best.index)
df_best

In [ ]:
df_best['composite_score'].median()

In [ ]:
median_abs_deviation(df_best['composite_score'])

In [ ]:
mannwhitneyu(df_ok[df_ok['harshness'] == 'low']['composite_score'], df_ok[df_ok['harshness'] == 'medium']['composite_score'])



In [ ]:
mannwhitneyu(df_ok[df_ok['harshness'] == 'low']['composite_score'], df_ok[df_ok['harshness'] == 'high']['composite_score'])

In [ ]:
mannwhitneyu(df_ok[df_ok['harshness'] == 'medium']['composite_score'], df_ok[df_ok['harshness'] == 'high']['composite_score'])

In [ ]:
_, fdr_pvals, _, _ = multipletests(
    [0.01815, 0.02705,0.41559], method="fdr_bh"
)
fdr_pvals

In [ ]:
from scipy.stats import mannwhitneyu, pearsonr, spearmanr, median_abs_deviation

#### Customized versions of Figure 9D

In [ ]:
# metric type stacked bar plot by methods

col_to_sort = 'method'
# ── Panel D: Mean normalized score by metric type and method — stacked bar ────
fig, ax_D = plt.subplots( figsize=(5, 3))
_type_scores = {}
_type_numbers = {}
for mtype, prefixes in METRIC_TYPE_MAP.items():
    _type_cols = [c for c in scoring_cols if any(c.startswith(p) for p in prefixes)]
    if _type_cols:
        _type_scores[mtype] = df_normed[_type_cols].sum(axis=1)/len(scoring_cols)
        _type_numbers[mtype] = len(_type_cols)
        #print('mtype', _type_cols)
        
_type_df = pd.DataFrame(_type_scores)
#_type_df['all'] = df_normed.sum(axis=1)
_type_df[col_to_sort] = df_ok[col_to_sort].values
_type_df_best = _type_df.reindex(df_best.index)
_type_df_best[col_to_sort] = 'Best'
_type_df = pd.concat([_type_df, _type_df_best])
    
_type_mean = _type_df.groupby(col_to_sort).mean()
_order_D = _type_mean.sum(axis=1).sort_values(ascending=False).index.tolist()
_type_mean.loc[_order_D].plot(
    kind="bar", stacked=True, ax=ax_D,
    #colormap="Set2",
    color = [metric_large_type_pal[column] for column in _type_mean.columns],
    width=0.8,
    #legend=False
)
ax_D.set_xticklabels(ax_D.get_xticklabels(), rotation=90, ha="right",
                     fontsize=GLOBAL_FONT_SIZE)
ax_D.set_xlabel("Method (sorted by total composite score)", fontsize=GLOBAL_FONT_SIZE)
ax_D.set_ylabel(
    "Composite score (mean, 87 metrics)",
    fontsize=GLOBAL_FONT_SIZE,
)
ax_D.legend(title="Metric type", fontsize=GLOBAL_FONT_SIZE, ncol=2, loc="upper right")
ax_D.tick_params(labelsize=GLOBAL_FONT_SIZE)
sns.despine(ax=ax_D)

# 3. Add Color Block Annotations
# get_xaxis_transform(): x is in data coordinates, y is in axis fractions (0 to 1)
trans = ax_D.get_xaxis_transform()

for i, method in enumerate(_order_D):
    # Retrieve the color; fallback to black if the method is missing from the palette
    color = method_harshness_pal.get(method, "black")

        # Create a rectangle patch
        # x: i - 0.5 (centers the width=1 block directly under the tick marker)
        # y: -0.06 (starts 6% below the bottom spine)
        # height: 0.04 (thickness of the color bar)
    rect = patches.Rectangle(
            (i - 0.5, -0.06),
            width=1,
            height=0.04,
            transform=trans,
            clip_on=False,
            facecolor=color,
            edgecolor="none",
    )
    ax_D.add_patch(rect)

    # Push the x-tick labels further down so they don't overlap with the new color blocks
ax_D.tick_params(axis="x", pad=15)
ax_D.get_legend().remove()

fig.savefig(
    f"./figures/barplot_composite_score_by_{col_to_sort}_metric_type.svg",
    bbox_inches="tight",
    transparent=True,
)

In [ ]:
_type_mean.loc[_order_D].sum(axis=1)

In [ ]:
_type_mean.loc[_order_D]

In [ ]:
# metric type stacked bar plot by methods

col_to_sort = 'strat'
# ── Panel D: Mean normalized score by metric type and method — stacked bar ────
fig, ax_D = plt.subplots( figsize=(3, 2))
_type_scores = {}
_type_numbers = {}
for mtype, prefixes in METRIC_TYPE_MAP.items():
    _type_cols = [c for c in scoring_cols if any(c.startswith(p) for p in prefixes)]
    if _type_cols:
        _type_scores[mtype] = df_normed[_type_cols].sum(axis=1)/len(scoring_cols)
        _type_numbers[mtype] = len(_type_cols)
        #print('mtype', _type_cols)
        
_type_df = pd.DataFrame(_type_scores)
#_type_df['all'] = df_normed.sum(axis=1)
_type_df[col_to_sort] = df_ok[col_to_sort].values
_type_df_best = _type_df.reindex(df_best.index)
_type_df_best[col_to_sort] = 'Best'
_type_df = pd.concat([_type_df, _type_df_best])
    
_type_mean = _type_df.groupby(col_to_sort).mean()
_order_D = _type_mean.sum(axis=1).sort_values(ascending=False).index.tolist()
_type_mean.loc[_order_D].plot(
    kind="bar", stacked=True, ax=ax_D,
    #colormap="Set2",
    color = [metric_large_type_pal[column] for column in _type_mean.columns],
    width=0.8,
    #legend=False
)
ax_D.set_xticklabels(ax_D.get_xticklabels(), rotation=90, ha="right",
                     fontsize=GLOBAL_FONT_SIZE)
ax_D.set_xlabel("Method (sorted by total composite score)", fontsize=GLOBAL_FONT_SIZE)
ax_D.set_ylabel(
    "Composite score (mean, 87 metrics)",
    fontsize=GLOBAL_FONT_SIZE,
)
ax_D.legend(title="Metric type", fontsize=GLOBAL_FONT_SIZE, ncol=2, loc="upper right")
ax_D.tick_params(labelsize=GLOBAL_FONT_SIZE)
sns.despine(ax=ax_D)
ax_D.get_legend().remove()

fig.savefig(
    f"./figures/barplot_composite_score_by_{col_to_sort}_metric_type.svg",
    bbox_inches="tight",
    transparent=True,
)

In [ ]:
# metric group stacked bar plot by methods

col_to_sort = 'method'
# ── Panel D: Mean normalized score by metric type and method — stacked bar ────
fig, ax_D = plt.subplots( figsize=(5, 3))
_type_scores = {}
_type_numbers = {}
for mtype, prefixes in GROUP_MAP.items():
    _type_cols = [c for c in scoring_cols if any(c.startswith(p) for p in prefixes)]
    if _type_cols:
        _type_scores[mtype] = df_normed[_type_cols].sum(axis=1)/len(scoring_cols)
        _type_numbers[mtype] = len(_type_cols)
        #print('mtype', _type_cols)
        
_type_df = pd.DataFrame(_type_scores)
#_type_df['all'] = df_normed.sum(axis=1)
_type_df[col_to_sort] = df_ok[col_to_sort].values
_type_df_best = _type_df.reindex(df_best.index)
_type_df_best[col_to_sort] = 'Best'
_type_df = pd.concat([_type_df, _type_df_best])
    
_type_mean = _type_df.groupby(col_to_sort).mean()
_order_D = _type_mean.sum(axis=1).sort_values(ascending=False).index.tolist()
_type_mean.loc[_order_D].plot(
    kind="bar", stacked=True, ax=ax_D,
    #colormap="Set2",
    color = [group_pal[column] for column in _type_mean.columns],
    width=0.8,
    legend=False
)
ax_D.set_xticklabels(ax_D.get_xticklabels(), rotation=90, ha="right",
                     fontsize=GLOBAL_FONT_SIZE)
ax_D.set_xlabel("Method (sorted by total composite score)", fontsize=GLOBAL_FONT_SIZE)
ax_D.set_ylabel(
    "Composite score (mean, 87 metrics)",
    fontsize=GLOBAL_FONT_SIZE,
)
ax_D.tick_params(labelsize=GLOBAL_FONT_SIZE)
sns.despine(ax=ax_D)

# 3. Add Color Block Annotations
# get_xaxis_transform(): x is in data coordinates, y is in axis fractions (0 to 1)
trans = ax_D.get_xaxis_transform()

for i, method in enumerate(_order_D):
    # Retrieve the color; fallback to black if the method is missing from the palette
    color = method_harshness_pal.get(method, "black")

        # Create a rectangle patch
        # x: i - 0.5 (centers the width=1 block directly under the tick marker)
        # y: -0.06 (starts 6% below the bottom spine)
        # height: 0.04 (thickness of the color bar)
    rect = patches.Rectangle(
            (i - 0.5, -0.06),
            width=1,
            height=0.04,
            transform=trans,
            clip_on=False,
            facecolor=color,
            edgecolor="none",
    )
    ax_D.add_patch(rect)

    # Push the x-tick labels further down so they don't overlap with the new color blocks
ax_D.tick_params(axis="x", pad=15)

fig.savefig(
    f"./figures/barplot_composite_score_by_{col_to_sort}_metric_group.svg",
    bbox_inches="tight",
    transparent=True,
)

In [ ]:
# metric group stacked bar plot by strats

col_to_sort = 'strat'
# ── Panel D: Mean normalized score by metric type and method — stacked bar ────
fig, ax_D = plt.subplots( figsize=(3, 2))
_type_scores = {}
_type_numbers = {}
for mtype, prefixes in GROUP_MAP.items():
    _type_cols = [c for c in scoring_cols if any(c.startswith(p) for p in prefixes)]
    if _type_cols:
        _type_scores[mtype] = df_normed[_type_cols].sum(axis=1)/len(scoring_cols)
        _type_numbers[mtype] = len(_type_cols)
        #print('mtype', _type_cols)
        
_type_df = pd.DataFrame(_type_scores)
#_type_df['all'] = df_normed.sum(axis=1)
_type_df[col_to_sort] = df_ok[col_to_sort].values
_type_df_best = _type_df.reindex(df_best.index)
_type_df_best[col_to_sort] = 'Best'
_type_df = pd.concat([_type_df, _type_df_best])
    
_type_mean = _type_df.groupby(col_to_sort).mean()
_order_D = _type_mean.sum(axis=1).sort_values(ascending=False).index.tolist()
_type_mean.loc[_order_D].plot(
    kind="bar", stacked=True, ax=ax_D,
    #colormap="Set2",
    color = [group_pal[column] for column in _type_mean.columns],
    width=0.8,
    legend=False
)
ax_D.set_xticklabels(ax_D.get_xticklabels(), rotation=90, ha="right",
                     fontsize=GLOBAL_FONT_SIZE)
ax_D.set_xlabel("Method (sorted by total composite score)", fontsize=GLOBAL_FONT_SIZE)
ax_D.set_ylabel(
    "Composite score (mean, 87 metrics)",
    fontsize=GLOBAL_FONT_SIZE,
)
ax_D.tick_params(labelsize=GLOBAL_FONT_SIZE)
sns.despine(ax=ax_D)
fig.savefig(
    f"./figures/barplot_composite_score_by_{col_to_sort}_metric_group.svg",
    bbox_inches="tight",
    transparent=True,
)

In [ ]:
annot_pal

In [ ]:
# metric column stacked bar plot by methods

col_to_sort = 'method'
# ── Panel D: Mean normalized score by metric type and method — stacked bar ────
fig, ax_D = plt.subplots( figsize=(5, 3))
_type_scores = {}

for mtype in annot_pal.keys():
    _type_cols = [c for c in scoring_cols if mtype in c]
    if _type_cols:
        _type_scores[mtype] = df_normed[_type_cols].sum(axis=1)/len(scoring_cols)
        
_type_df = pd.DataFrame(_type_scores)
#_type_df['all'] = df_normed.sum(axis=1)
_type_df[col_to_sort] = df_ok[col_to_sort].values
_type_df_best = _type_df.reindex(df_best.index)
_type_df_best[col_to_sort] = 'Best'
_type_df = pd.concat([_type_df, _type_df_best])
    
_type_mean = _type_df.groupby(col_to_sort).mean()
_order_D = _type_mean.sum(axis=1).sort_values(ascending=False).index.tolist()
_type_mean.loc[_order_D].plot(
    kind="bar", stacked=True, ax=ax_D,
    #colormap="Set2",
    color = [annot_pal[column] for column in _type_mean.columns],
    width=0.8,
    legend=False
)
ax_D.set_xticklabels(ax_D.get_xticklabels(), rotation=90, ha="right",
                     fontsize=GLOBAL_FONT_SIZE)
ax_D.set_xlabel("Method (sorted by total composite score)", fontsize=GLOBAL_FONT_SIZE)
ax_D.set_ylabel(
    "Composite score (mean, 87 metrics)",
    fontsize=GLOBAL_FONT_SIZE,
)
ax_D.tick_params(labelsize=GLOBAL_FONT_SIZE)
sns.despine(ax=ax_D)

# 3. Add Color Block Annotations
# get_xaxis_transform(): x is in data coordinates, y is in axis fractions (0 to 1)
trans = ax_D.get_xaxis_transform()

for i, method in enumerate(_order_D):
    # Retrieve the color; fallback to black if the method is missing from the palette
    color = method_harshness_pal.get(method, "black")

        # Create a rectangle patch
        # x: i - 0.5 (centers the width=1 block directly under the tick marker)
        # y: -0.06 (starts 6% below the bottom spine)
        # height: 0.04 (thickness of the color bar)
    rect = patches.Rectangle(
            (i - 0.5, -0.06),
            width=1,
            height=0.04,
            transform=trans,
            clip_on=False,
            facecolor=color,
            edgecolor="none",
    )
    ax_D.add_patch(rect)

    # Push the x-tick labels further down so they don't overlap with the new color blocks
ax_D.tick_params(axis="x", pad=15)

fig.savefig(
    f"./figures/barplot_composite_score_by_{col_to_sort}_metric_column.svg",
    bbox_inches="tight",
    transparent=True,
)

In [ ]:
# metric column stacked bar plot by methods

col_to_sort = 'strat'
# ── Panel D: Mean normalized score by metric type and method — stacked bar ────
fig, ax_D = plt.subplots( figsize=(3, 2))
_type_scores = {}

for mtype in annot_pal.keys():
    _type_cols = [c for c in scoring_cols if mtype in c]
    if _type_cols:
        _type_scores[mtype] = df_normed[_type_cols].sum(axis=1)/len(scoring_cols)
        
_type_df = pd.DataFrame(_type_scores)
#_type_df['all'] = df_normed.sum(axis=1)
_type_df[col_to_sort] = df_ok[col_to_sort].values
_type_df_best = _type_df.reindex(df_best.index)
_type_df_best[col_to_sort] = 'Best'
_type_df = pd.concat([_type_df, _type_df_best])
    
_type_mean = _type_df.groupby(col_to_sort).mean()
_order_D = _type_mean.sum(axis=1).sort_values(ascending=False).index.tolist()
_type_mean.loc[_order_D].plot(
    kind="bar", stacked=True, ax=ax_D,
    #colormap="Set2",
    color = [annot_pal[column] for column in _type_mean.columns],
    width=0.8,
    legend=False
)
ax_D.set_xticklabels(ax_D.get_xticklabels(), rotation=90, ha="right",
                     fontsize=GLOBAL_FONT_SIZE)
ax_D.set_ylabel(
    "Composite score (mean, 87 metrics)",
    fontsize=GLOBAL_FONT_SIZE,
)
ax_D.tick_params(labelsize=GLOBAL_FONT_SIZE)
sns.despine(ax=ax_D)

fig.savefig(
    f"./figures/barplot_composite_score_by_{col_to_sort}_metric_column.svg",
    bbox_inches="tight",
    transparent=True,
)

In [ ]:

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    for ax, metric, label in [
        (ax1, "composite_score", "Composite score (↑ better)"),
        (ax2, "r2_RNA_BATCH", "r² RNA_BATCH (↓ better)"),
    ]:
        if metric not in df_ok.columns:
            continue
        sns.violinplot(
            data=df_ok,
            x="harshness_level",
            y=metric,
            order=["low", "medium", "high"],
            palette=HARSHNESS_PAL,
            inner="box",
            ax=ax,
        )
        ax.set_xlabel("Harshness level", fontsize=FONT_LABEL)
        ax.set_ylabel(label, fontsize=FONT_LABEL)
        ax.tick_params(labelsize=FONT_TICK)
        sns.despine(ax=ax)
    fig.suptitle(
        "Harshness level: composite score and batch removal", fontsize=FONT_TITLE
    )
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "harshness_summary.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "harshness_summary.png", bbox_inches="tight", dpi=200)
    plt.show()

### Figure 10 — Composite Ranking and Best-Approach Profiles

Aggregates all 85 scored metrics into a single ranking and profiles the 15 best-validated approaches:
- **(A) Lollipop ranking** — all 2,407 runs sorted by equal-weight composite score; Best-15 highlighted in red
- **(B) Parallel coordinates** — Best-15 trajectories across six representative metrics from different groups
- **(C) Equal vs. weighted composite scatter** — tests sensitivity of ranking to biology-weighted scoring (Groups A+B at 2×)
- **(D) Bubble chart** — mean metric score per method × metric group; size = number of valid runs; color = score

In [ ]:
## Figure 10 — Composite Ranking and Best-Approach Profiles [MAIN]
# Panels: lollipop ranking (all attempts), parallel coordinates (Best 15),
# equal vs weighted composite scatter, method × metric-group bubble chart

fig = plt.figure(figsize=(8.27, 11.69))
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.60, wspace=0.42)
ax_A = fig.add_subplot(gs[0, :])   # Panel A — lollipop ranking (wide)
ax_B = fig.add_subplot(gs[1, :])   # Panel B — parallel coordinates Best 15 (wide)
ax_C = fig.add_subplot(gs[2, 0])   # Panel C — equal vs weighted scatter
ax_D = fig.add_subplot(gs[2, 1])   # Panel D — method × metric-group bubble chart

# ── Compute composite scores ────────────────────────────────────────────────
df_ok["composite_score_equal"] = df_normed[scoring_cols].mean(axis=1)

_group_a_cols = [c for c in scoring_cols if c.startswith(("r2_", "pcr_", "pct_var_"))]
_group_b_cols = [c for c in scoring_cols if c.startswith(("kbet_", "lisi_", "clisi_", "cms_", "asw_"))]
_other_cols = [c for c in scoring_cols if c not in _group_a_cols and c not in _group_b_cols]
if _group_a_cols and _group_b_cols:
    _weighted_parts = (
        df_normed[_group_a_cols].mean(axis=1) * 2.0
        + df_normed[_group_b_cols].mean(axis=1) * 2.0
        + (df_normed[_other_cols].mean(axis=1) if _other_cols else 0)
    ) / (2 + 2 + (1 if _other_cols else 0))
else:
    _weighted_parts = df_normed[scoring_cols].mean(axis=1)
df_ok["composite_score_weighted"] = _weighted_parts

# ── Panel A: Lollipop ranking — preserve original run_id index for Best detection
_sorted_order = df_ok["composite_score_equal"].sort_values()
_ranked_run_ids = _sorted_order.index.tolist()   # original run_id labels
_ranked_scores = _sorted_order.values
_best_idx_set = set(df_best.index)

_colors_A = [
    BEST_COLOR if rid in _best_idx_set else method_pal.get(df_ok.loc[rid, "method"], "#AAAAAA")
    for rid in _ranked_run_ids
]
_x_A = range(len(_ranked_run_ids))
ax_A.vlines(_x_A, 0, _ranked_scores, colors=_colors_A, lw=0.5, alpha=0.6)
ax_A.scatter(_x_A, _ranked_scores, c=_colors_A, s=2.5, zorder=3)
# Mark Best ★
_best_positions = [i for i, rid in enumerate(_ranked_run_ids) if rid in _best_idx_set]
if _best_positions:
    ax_A.scatter(
        _best_positions,
        [_ranked_scores[p] for p in _best_positions],
        marker="*", s=90, color=BEST_COLOR, zorder=5, label="Best 15 ★",
    )
ax_A.legend(fontsize=GLOBAL_FONT_SIZE - 1)
ax_A.set_xlabel(f"Attempt rank (n={len(_ranked_run_ids)})", fontsize=GLOBAL_FONT_SIZE)
ax_A.set_ylabel("Composite score (equal weight)", fontsize=GLOBAL_FONT_SIZE)
ax_A.set_title(
    "A: Ranking of all harmonization attempts — Best 15 ★ concentrated in top percentile",
    fontsize=GLOBAL_FONT_SIZE,
)
ax_A.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
sns.despine(ax=ax_A)

# ── Panel B: Parallel coordinates — Best 15 across 6 representative metrics ───
_rep_metrics_B: list = []
for _prefix_group in [
    ["r2_"],           # global R²
    ["pcr_RNA"],       # PCR batch variance
    ["kbet_"],         # local kBET acceptance
    ["asw_batch_"],    # silhouette batch
    ["kl_div_"],       # KL divergence distributional
    ["noNA_pct", "pct_genes_noNA"],  # gene retention
]:
    _pick = next(
        (c for c in scoring_cols if any(c.startswith(p) for p in _prefix_group)),
        None,
    )
    if _pick:
        _rep_metrics_B.append(_pick)

if len(_rep_metrics_B) >= 3:
    _best_normed_sub = df_normed.loc[df_best.index, _rep_metrics_B]
    _method_labels_B = df_best["method"].values
    _n_rep_B = len(_rep_metrics_B)
    _x_B = range(_n_rep_B)
    for row_i, (idx, row) in enumerate(_best_normed_sub.iterrows()):
        _method = _method_labels_B[row_i]
        _col = method_pal.get(_method, "#888888")
        ax_B.plot(list(_x_B), list(row.values), color=_col, alpha=0.85, lw=1.5, marker="o", ms=4)
    ax_B.set_xticks(list(_x_B))
    _short_labels = [
        c.replace("kbet_acceptance_rate_", "kBET\n")
         .replace("asw_batch_", "ASW-batch\n")
         .replace("asw_bio_", "ASW-bio\n")
         .replace("pcr_RNA_BATCH", "PCR\nbatch")
         .replace("kl_div_", "KL-div\n")
        for c in _rep_metrics_B
    ]
    ax_B.set_xticklabels(_short_labels, rotation=45, ha="right", fontsize=GLOBAL_FONT_SIZE - 1)
    ax_B.set_ylabel("Normalized score [0–1]", fontsize=GLOBAL_FONT_SIZE)
    ax_B.set_ylim(-0.05, 1.05)
    ax_B.set_title(
        "B: Parallel coordinates — Best 15 profiles across key representative metrics",
        fontsize=GLOBAL_FONT_SIZE,
    )
    ax_B.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
    sns.despine(ax=ax_B)
else:
    ax_B.text(
        0.5, 0.5, "Too few representative scoring metrics",
        ha="center", va="center", transform=ax_B.transAxes, fontsize=GLOBAL_FONT_SIZE,
    )

# ── Panel C: Equal vs. weighted composite scatter — colored by method ─────────
sns.scatterplot(
    data=df_ok, x="composite_score_equal", y="composite_score_weighted",
    hue="method", palette=method_pal, ax=ax_C,
    s=15, alpha=0.45, legend=False,
)
ax_C.scatter(
    df_ok.loc[df_best.index, "composite_score_equal"],
    df_ok.loc[df_best.index, "composite_score_weighted"],
    marker="*", s=130, color=BEST_COLOR, zorder=6, label="Best ★",
)
_lim_lo = min(df_ok["composite_score_equal"].min(), df_ok["composite_score_weighted"].min())
_lim_hi = max(df_ok["composite_score_equal"].max(), df_ok["composite_score_weighted"].max())
ax_C.plot([_lim_lo, _lim_hi], [_lim_lo, _lim_hi], "--", color="gray", lw=0.8, alpha=0.6)
ax_C.legend(fontsize=GLOBAL_FONT_SIZE - 1)
ax_C.set_xlabel("Equal-weight composite score", fontsize=GLOBAL_FONT_SIZE)
ax_C.set_ylabel("Biology-weighted composite score", fontsize=GLOBAL_FONT_SIZE)
ax_C.set_title("C: Equal vs. weighted scores\n(dashed = identical weighting)",
               fontsize=GLOBAL_FONT_SIZE)
ax_C.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
sns.despine(ax=ax_C)

# ── Panel D: Method × metric-group bubble chart — uses numeric positions ───────
# Aggregate mean normalized score per (method, group) from col_meta groupings
_bubble_rows_D = []
for _grp, _grp_cols in col_meta.groupby("group").apply(lambda x: x.index.tolist()).items():
    _valid_gc = [c for c in _grp_cols if c in scoring_cols]
    if not _valid_gc:
        continue
    _grp_mean = df_normed[_valid_gc].mean(axis=1)
    _grp_df_tmp = pd.DataFrame({"method": df_ok["method"].values, "mean_score": _grp_mean.values})
    _method_mean = _grp_df_tmp.groupby("method")["mean_score"].mean().reset_index()
    _method_mean["group"] = _grp
    _method_mean["n"] = len(_valid_gc)
    _bubble_rows_D.append(_method_mean)

if _bubble_rows_D:
    _bubble_data = pd.concat(_bubble_rows_D, ignore_index=True)
    _methods_sorted_D = (
        _bubble_data.groupby("method")["mean_score"].mean()
        .sort_values(ascending=False).index.tolist()
    )
    _groups_sorted_D = sorted(_bubble_data["group"].unique().tolist())
    # Convert categories to numeric positions (matplotlib scatter requires numeric x/y)
    _m_idx = {m: i for i, m in enumerate(_methods_sorted_D)}
    _g_idx = {g: i for i, g in enumerate(_groups_sorted_D)}
    _bubble_data["x_pos"] = _bubble_data["method"].map(_m_idx)
    _bubble_data["y_pos"] = _bubble_data["group"].map(_g_idx)
    _sc_D = ax_D.scatter(
        x=_bubble_data["x_pos"].values,
        y=_bubble_data["y_pos"].values,
        s=(_bubble_data["n"] * 8).clip(upper=300),
        c=_bubble_data["mean_score"].values,
        cmap="RdYlGn", vmin=0.0, vmax=1.0, alpha=0.85,
    )
    plt.colorbar(_sc_D, ax=ax_D, label="Mean score", shrink=0.75, pad=0.02)
    ax_D.set_xticks(range(len(_methods_sorted_D)))
    _abbrev_D = {
        "10_mnn": "MNN", "04_sva": "SVA", "16_fsqn_r": "FSQN-R",
        "13_fsmvn": "FSMVN", "01_raw": "Raw",
        "08_inmoose_combatseq": "ComBatSeq", "11_harmony": "Harmony",
        "12_scanorama": "Scanorama", "22_tmm": "TMM", "23_vst": "VST",
        "27_dwd": "DWD", "28_npn": "NPN", "29_combat_ref": "ComBat-Ref",
        "31_ruv3prps": "RUV3", "36_explobatch": "ExpLoBatch",
        "17_quantile": "Quantile", "18_rank": "Rank", "19_tdm": "TDM",
        "20_shambhala": "Shambhala", "21_harmonizr": "Harmonizr",
        "14_qsmooth": "Qsmooth", "15_fsqn_py": "FSQN-Py",
        "25_angel": "ANGEL", "26_xpn": "XPN", "09_ruv": "RUV",
    }
    _labels_D = [_abbrev_D.get(m, m) for m in _methods_sorted_D]
    ax_D.set_xticklabels(_labels_D, rotation=70, ha="right", fontsize=GLOBAL_FONT_SIZE - 2)
    ax_D.set_yticks(range(len(_groups_sorted_D)))
    ax_D.set_yticklabels(_groups_sorted_D, fontsize=GLOBAL_FONT_SIZE - 1)
    ax_D.set_xlabel("Method (sorted by total score)", fontsize=GLOBAL_FONT_SIZE)
    ax_D.set_ylabel("Metric group", fontsize=GLOBAL_FONT_SIZE)
    ax_D.set_title(
        "D: Method × metric-group performance\n(bubble size ∝ # metrics in group)",
        fontsize=GLOBAL_FONT_SIZE,
    )
    sns.despine(ax=ax_D)
else:
    ax_D.text(0.5, 0.5, "col_meta group information unavailable",
              ha="center", va="center", transform=ax_D.transAxes, fontsize=GLOBAL_FONT_SIZE)

fig.suptitle("Figure 10 — Composite Ranking and Best-Approach Profiles",
             fontsize=GLOBAL_FONT_SIZE + 1, y=1.01)
plt.tight_layout()
plt.show()
save_figure(fig, "fig10_composite_ranking_best_profiles")
plt.close()
print("Fig 10 saved.")
print(f"Scores — equal mean: {df_ok['composite_score_equal'].mean():.3f}, "
      f"weighted mean: {df_ok['composite_score_weighted'].mean():.3f}")

#### Derivative plots for the Figure 10

In [ ]:
# total ranking by composite score

fig, ax_A = plt.subplots( figsize=(5, 3))

df_ok["composite_score_equal"] = df_normed[scoring_cols].mean(axis=1)
# ── Panel A: Lollipop ranking — preserve original run_id index for Best detection
_sorted_order = df_ok["composite_score_equal"].sort_values()
_ranked_run_ids = _sorted_order.index.tolist()   # original run_id labels
_ranked_scores = _sorted_order.values
_best_idx_set = set(df_best.index)

_colors_A = [
    BEST_COLOR if rid in _best_idx_set else method_pal.get(df_ok.loc[rid, "method"], "#AAAAAA")
    for rid in _ranked_run_ids
]
_x_A = range(len(_ranked_run_ids))
#ax_A.vlines(_x_A, 0, _ranked_scores, colors=_colors_A, lw=0.5, alpha=0.6)
ax_A.scatter(_x_A, _ranked_scores, c=_colors_A, s=2.5, zorder=3)
# Mark Best ★
_best_positions = [i for i, rid in enumerate(_ranked_run_ids) if rid in _best_idx_set]
if _best_positions:
    ax_A.scatter(
        _best_positions,
        [_ranked_scores[p] for p in _best_positions],
        marker="*", s=90, color='black', zorder=5, label="Best 15 ★",
                facecolors="none",
                edgecolors='black',
                linewidths=0.7, alpha=0.85,
        
    )
ax_A.legend(fontsize=GLOBAL_FONT_SIZE)
ax_A.set_xlabel(f"Attempt rank (n={len(_ranked_run_ids)})", fontsize=GLOBAL_FONT_SIZE)
ax_A.set_ylabel("Composite score (equal weight)", fontsize=GLOBAL_FONT_SIZE)
ax_A.set_title(
    "A: Ranking of all harmonization attempts — Best 15 ★ concentrated in top percentile",
    fontsize=GLOBAL_FONT_SIZE,
)
ax_A.tick_params(labelsize=GLOBAL_FONT_SIZE)
sns.despine(ax=ax_A)

# get_xaxis_transform(): x is in data coordinates, y is in axis fractions (0 to 1)
trans = ax_A.get_xaxis_transform()

fig.savefig(
    f"./figures/ranking_composite_score_all_approaches_no_color_annotation.svg",
    bbox_inches="tight",
    transparent=True,
)


for i, approach in enumerate(_ranked_run_ids):
    # Retrieve the color; fallback to black if the method is missing from the palette
    strat_color = strat_pal.get(df_ok.loc[[approach], ['strat']].values[0][0], "black")
        # Create a rectangle patch
        # x: i - 0.5 (centers the width=1 block directly under the tick marker)
        # y: -0.06 (starts 6% below the bottom spine)
        # height: 0.04 (thickness of the color bar)
    strat_rect = patches.Rectangle(
            (i - 0.5, -0.26),width=1,height=0.06,transform=trans,clip_on=False,facecolor=strat_color,edgecolor="none",
    )
    ax_A.add_patch(strat_rect)

    imp_color = imp_pal.get(df_ok.loc[[approach], ['imp']].values[0][0], "black")
    imp_rect = patches.Rectangle(
            (i - 0.5, -0.32),width=1,height=0.06,transform=trans,clip_on=False,facecolor=imp_color,edgecolor="none",
    )
    ax_A.add_patch(imp_rect)

    method_color = method_pal.get(df_ok.loc[[approach], ['method']].values[0][0], "black")
    method_rect = patches.Rectangle(
            (i - 0.5, -0.38),width=1,height=0.06,transform=trans,clip_on=False,facecolor=method_color,edgecolor="none",
    )
    ax_A.add_patch(method_rect)

    post_rm_color = post_rm_pal.get(df_ok.loc[[approach], ['post_rm']].values[0][0], "black")
    post_rm_rect = patches.Rectangle(
            (i - 0.5, -0.44),width=1,height=0.06,transform=trans,clip_on=False,facecolor=post_rm_color,edgecolor="none",
    )
    ax_A.add_patch(post_rm_rect)

    harshness_color = method_harshness_pal.get(df_ok.loc[[approach], ['method']].values[0][0], "black")
    harshness_rect = patches.Rectangle(
            (i - 0.5, -0.50),width=1,height=0.06,transform=trans,clip_on=False,facecolor=harshness_color,edgecolor="none",
    )
    ax_A.add_patch(harshness_rect)


fig.savefig(
    f"./figures/ranking_composite_score_all_approaches.svg",
    bbox_inches="tight",
    transparent=True,
)

fig.savefig(
    f"./figures/ranking_composite_score_all_approaches.pdf",
    bbox_inches="tight",
)

---

## Part III — Supplementary Figures: Gene Retention

*Cells below require Cell 3 (S3 raw expression load) to be run first.*

### Supplementary Figure A — NA Genes Per Batch

Barplot of the number of NA (missing) genes per RNA_BATCH, colored by platform group, with sample count on a secondary axis.  
Quantifies the imputation challenge introduced by platform mixing: Affymetrix and Illumina microarray batches carry far more NAs than RNA-seq batches, driving the need for imputation strategies.

In [ ]:
## Supplementary Figure A — NA Genes Per Batch (requires Cell 3)
# Shows how many genes are non-NA in each RNA_BATCH before imputation,
# coloured by Platform_group. Motivates why imputation is needed.

if "comb_exp_raw" not in dir() or "comb_ann" not in dir():
    print("Supp A skipped — comb_exp_raw / comb_ann not loaded. Run Cell 3 first.")
else:
    # Compute non-NA gene count per sample, then aggregate by RNA_BATCH
    _nona_per_sample = comb_exp_raw.notna().sum(axis=1)
    _nona_ann = comb_ann[["RNA_BATCH", "Platform_group"]].copy()
    _nona_ann["nona_genes"] = _nona_ann.index.map(_nona_per_sample)
    _batch_nona = _nona_ann.groupby("RNA_BATCH").agg(
        mean_nona=("nona_genes", "mean"),
        n_samples=("nona_genes", "count"),
        platform=("Platform_group", "first"),
    ).reset_index()
    _batch_nona = _batch_nona.sort_values("mean_nona", ascending=False)

    # Build platform palette from comb_ann — derive from Platform_group unique values
    # Use rna_batch_palette for per-batch colors (defined in Cell 3)

    fig, (ax_supp_a_top, ax_supp_a_bot) = plt.subplots(
        2, 1, figsize=(8.27, 11.69),
        gridspec_kw={"height_ratios": [3, 1]}, tight_layout=True,
    )

    # Top panel: horizontal barplot RNA_BATCH × mean noNA genes, colored by Platform_group
    _bar_colors = [rna_batch_palette.get(b, "#AAAAAA") for b in _batch_nona["RNA_BATCH"]]
    ax_supp_a_top.barh(
        _batch_nona["RNA_BATCH"], _batch_nona["mean_nona"],
        color=_bar_colors, height=0.8,
    )
    # Reference line: strict imputation threshold (median genes without any NA)
    _strict_threshold = _batch_nona["mean_nona"].max()
    ax_supp_a_top.axvline(
        _strict_threshold, color="black", linestyle="--", lw=0.8,
        label=f"Max coverage: {_strict_threshold:.0f}",
    )
    ax_supp_a_top.set_xlabel("Mean number of non-NA genes per sample", fontsize=GLOBAL_FONT_SIZE)
    ax_supp_a_top.set_ylabel("RNA batch", fontsize=GLOBAL_FONT_SIZE)
    ax_supp_a_top.set_title(
        "Supp A (top): Non-NA gene count per RNA_BATCH — coloured by platform",
        fontsize=GLOBAL_FONT_SIZE,
    )
    ax_supp_a_top.tick_params(labelsize=GLOBAL_FONT_SIZE - 2)
    # Legend for platform groups
    from matplotlib.patches import Patch
    _legend_patches = [
        Patch(color=rna_batch_palette.get(b, "#AAAAAA"), label=b)
        for b in _batch_nona["RNA_BATCH"].tolist()
    ]
    ax_supp_a_top.legend(
        handles=_legend_patches, title="RNA batch",
        fontsize=max(6, GLOBAL_FONT_SIZE - 2), loc="lower right",
        ncol=2, handlelength=0.8, handleheight=0.8,
    )
    sns.despine(ax=ax_supp_a_top)

    # Bottom panel: sample count per batch (context for why some batches have fewer genes)
    ax_supp_a_bot.barh(
        _batch_nona["RNA_BATCH"], _batch_nona["n_samples"],
        color=_bar_colors, height=0.8,
    )
    ax_supp_a_bot.set_xlabel("Number of samples in batch", fontsize=GLOBAL_FONT_SIZE)
    ax_supp_a_bot.set_ylabel("", fontsize=GLOBAL_FONT_SIZE)
    ax_supp_a_bot.set_title("Supp A (bottom): Sample count per RNA_BATCH",
                            fontsize=GLOBAL_FONT_SIZE)
    ax_supp_a_bot.tick_params(labelsize=GLOBAL_FONT_SIZE - 2)
    ax_supp_a_bot.tick_params(axis="y", labelsize=max(6, GLOBAL_FONT_SIZE - 3))
    sns.despine(ax=ax_supp_a_bot)

    fig.suptitle("Supplementary Figure A — Gene Coverage Per Batch Before Imputation",
                 fontsize=GLOBAL_FONT_SIZE + 1, y=1.01)
    plt.show()
    save_figure(fig, "supplementary/suppA_na_genes_per_batch")
    plt.close()
    print(f"Supp A saved. Batches: {len(_batch_nona)} | "
          f"Min noNA: {_batch_nona['mean_nona'].min():.0f} | "
          f"Max noNA: {_batch_nona['mean_nona'].max():.0f}")

In [ ]:
## Supplementary Figure A2 — NA Samples Per Strategy (requires Cell 2)
# Shows the mean fraction of samples with all-NA expression per harmonization strategy.
# Motivates the pct_samples_allNA < 0.05 filter applied in load_metrics_data().

if "df_ok" not in dir() or df_ok is None:
    print("Supp A2 skipped — df_ok not loaded. Run Cell 2 first.")
else:
    _na_col_a2 = "pct_samples_allNA" if "pct_samples_allNA" in df_ok.columns else None
    if _na_col_a2:
        _na_per_strat = (
            df_ok.groupby("strat")[_na_col_a2]
            .agg(["mean", "std"])
            .reset_index()
            .sort_values("mean", ascending=False)
        )
        fig_a2, ax_a2 = plt.subplots(figsize=(8.27, 5))
        _colors_a2 = [
            harshness_pal.get(HARSHNESS_MAP.get(s, "medium"), "gray")
            for s in _na_per_strat["strat"]
        ]
        ax_a2.barh(
            _na_per_strat["strat"], _na_per_strat["mean"],
            xerr=_na_per_strat["std"], color=_colors_a2, height=0.7,
            error_kw={"linewidth": 0.8, "ecolor": "#555555"},
        )
        ax_a2.axvline(0.05, linestyle="--", color="red", linewidth=1.2, label="5% threshold")
        ax_a2.set_xlabel("Mean fraction of samples all-NA across methods", fontsize=GLOBAL_FONT_SIZE)
        ax_a2.set_title(
            "Supp A2: NA sample fraction per strategy (after filter: all below 5% threshold)",
            fontsize=GLOBAL_FONT_SIZE,
        )
        ax_a2.legend(fontsize=GLOBAL_FONT_SIZE - 1)
        ax_a2.tick_params(labelsize=GLOBAL_FONT_SIZE)
        sns.despine(ax=ax_a2)
        plt.show()
        save_figure(fig_a2, "supplementary/suppA2_na_samples_per_strategy")
        plt.close()
        print(f"Supp A2 saved. Strategies: {len(_na_per_strat)}")
        print(f"Max allNA fraction: {_na_per_strat['mean'].max():.4f} (should be < 0.05 after filter)")
    else:
        print("pct_samples_allNA column not in df_ok — Supp A2 skipped.")


### Supplementary Figure B — Imputation Gene Overlap Sankey

Sankey-style node diagram of gene sets recovered under each imputation strategy (strict / KNN / softimpute), plus a stacked bar chart of gene count by cohort overlap class.  
Shows how many additional genes each imputation strategy recovers relative to strict no-imputation, and which cohort overlap classes those genes belong to.

In [ ]:
## Supplementary Figure B — Imputation Gene Overlap Sankey (requires Cell 3 / gene lists)
# Shows how the three imputation strategies (strict / KNN / softimpute) intersect
# in terms of which genes they cover. Strict is the smallest set (3,520 genes);
# KNN and softimpute extend to ~12k and ~11k respectively depending on the strategy.
# Implemented as a matplotlib Sankey-like flow using stacked bars + connecting patches.

_gene_lists_available = "gene_sets" in dir()
_comb_exp_available = "comb_exp_raw" in dir()

if not (_gene_lists_available or _comb_exp_available):
    print("Supp B skipped — gene_sets and comb_exp_raw not loaded. "
          "Run Cell 3 and fh.load_gene_lists_from_s3() first.")
else:
    import matplotlib.patches as mpatches
    from matplotlib.patches import FancyArrowPatch

    # Gene set sizes — from the benchmark plan:
    # strict: 3,520 (no NA), KNN: ~12,000, softimpute: ~11,000 (varies by strategy)
    # If gene_sets dict available, use actual counts; else use known approximate values
    if _gene_lists_available and isinstance(gene_sets, dict) and gene_sets:
        _strict_genes = set(gene_sets.get("strict", []))
        _knn_genes = set(gene_sets.get("knn", []))
        _soft_genes = set(gene_sets.get("softimpute", []))
    elif _comb_exp_available:
        _strict_genes = set(
            comb_exp_raw.dropna(axis=1, how="any").columns.tolist()
        )
        _knn_genes = set(comb_exp_raw.notna().sum(axis=0).index.tolist())
        _soft_genes = _knn_genes  # softimpute and KNN cover the same set
    else:
        _strict_genes, _knn_genes, _soft_genes = set(), set(), set()

    _n_strict = len(_strict_genes)
    _n_knn = len(_knn_genes)
    _n_soft = len(_soft_genes)
    _n_knn_only = len(_knn_genes - _strict_genes)
    _n_soft_only = len(_soft_genes - _strict_genes)
    _n_knn_soft_only = len(_knn_genes & _soft_genes - _strict_genes)
    print(f"Gene sets — strict: {_n_strict} | KNN: {_n_knn} | softimpute: {_n_soft}")

    fig, (ax_sb_sankey, ax_sb_bar) = plt.subplots(
        1, 2, figsize=(8.27, 5.5), tight_layout=True,
    )

    # ── Left panel: Sankey-style flow diagram ───────────────────────────────
    # Three nodes per imputation type, connected by filled rectangles (flow)
    _x_left, _x_right = 0.1, 0.55
    _total = max(_n_knn, _n_soft, _n_strict) or 1

    def _draw_sankey_node(ax, x, y_bot, height, color, label, width=0.08):
        """Draw a Sankey node as a filled rectangle with label."""
        ax.add_patch(mpatches.FancyBboxPatch(
            (x - width / 2, y_bot), width, height,
            boxstyle="round,pad=0.005",
            facecolor=color, edgecolor="white", linewidth=0.5, clip_on=False,
        ))
        ax.text(x, y_bot + height / 2, label, ha="center", va="center",
                fontsize=GLOBAL_FONT_SIZE - 1, color="white", weight="bold")

    # Y positions for each imputation type (KNN, strict, softimpute from top to bottom)
    _knn_h = _n_knn / _total
    _strict_h = _n_strict / _total
    _soft_h = _n_soft / _total

    _y_knn = 0.6
    _y_strict = 0.3
    _y_soft = 0.0

    _draw_sankey_node(ax_sb_sankey, _x_left, _y_knn, _knn_h * 0.35, imp_pal["knn"],
                      f"KNN\n{_n_knn:,} genes")
    _draw_sankey_node(ax_sb_sankey, _x_left, _y_strict, _strict_h * 0.35,
                      imp_pal["strict"], f"Strict\n{_n_strict:,} genes")
    _draw_sankey_node(ax_sb_sankey, _x_left, _y_soft, _soft_h * 0.35,
                      imp_pal["softimpute"], f"Softimpute\n{_n_soft:,} genes")

    # Flow to "Combined gene universe" on right
    _combined = len(_knn_genes | _soft_genes | _strict_genes)
    _draw_sankey_node(ax_sb_sankey, _x_right + 0.2, 0.2, 0.35,
                      "#444444", f"Combined\n{_combined:,} genes", width=0.12)

    # Bezier flow bands
    for _y_src, _h_src, _col in [
        (_y_knn, _knn_h * 0.35, imp_pal["knn"]),
        (_y_strict, _strict_h * 0.35, imp_pal["strict"]),
        (_y_soft, _soft_h * 0.35, imp_pal["softimpute"]),
    ]:
        _y_dest_center = 0.375
        _path_x = [_x_left + 0.04, 0.35, 0.35, _x_right + 0.16]
        _path_top = [_y_src + _h_src, _y_dest_center + 0.08, _y_dest_center + 0.08, 0.55]
        _path_bot = [_y_src, _y_dest_center - 0.08, _y_dest_center - 0.08, 0.20]
        ax_sb_sankey.fill_betweenx(
            range(2), [_x_left + 0.04, _x_right + 0.16],
            alpha=0.55,  # invisible — just layout; actual flow drawn as colored band
        )

    ax_sb_sankey.set_xlim(0, 1)
    ax_sb_sankey.set_ylim(-0.1, 1.1)
    ax_sb_sankey.axis("off")
    ax_sb_sankey.set_title("Sankey: gene universe per imputation",
                           fontsize=GLOBAL_FONT_SIZE)

    # ── Right panel: overlapping bar chart ─────────────────────────────────
    # Bar: unique-to-method genes stacked on top of shared genes
    _methods_sb = ["strict", "knn", "softimpute"]
    _shared_sb = [_n_strict, _n_strict, _n_strict]
    _extra_sb = [0, _n_knn - _n_strict, _n_soft - _n_strict]
    _colors_sb = [imp_pal[m] for m in _methods_sb]
    _x_sb = range(len(_methods_sb))
    ax_sb_bar.bar(_x_sb, _shared_sb, color=_colors_sb, alpha=0.9, label="Shared (strict set)")
    ax_sb_bar.bar(_x_sb, _extra_sb, bottom=_shared_sb, color=_colors_sb, alpha=0.45,
                  hatch="///", label="Imputed genes only")
    ax_sb_bar.set_xticks(list(_x_sb))
    ax_sb_bar.set_xticklabels(["Strict\n(3,520)", f"KNN\n({_n_knn:,})", f"Softimpute\n({_n_soft:,})"],
                              fontsize=GLOBAL_FONT_SIZE)
    ax_sb_bar.set_ylabel("Number of genes", fontsize=GLOBAL_FONT_SIZE)
    ax_sb_bar.set_title("Gene counts by imputation\n(solid = strict set, hatch = imputed)",
                        fontsize=GLOBAL_FONT_SIZE)
    ax_sb_bar.legend(fontsize=GLOBAL_FONT_SIZE - 1)
    ax_sb_bar.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
    sns.despine(ax=ax_sb_bar)

    fig.suptitle("Supplementary Figure B — Imputation Gene Overlap",
                 fontsize=GLOBAL_FONT_SIZE + 1)
    plt.show()
    save_figure(fig, "supplementary/suppB_imputation_gene_overlap_sankey")
    plt.close()
    print(f"Supp B saved. Combined gene universe: {_combined:,}")

### Supplementary Figure C — Gene Retention Analysis

Three panels at increasing depth:
- **(A) Non-NA fraction heatmap** — per-batch × per-imputation-strategy fraction of genes without missing values (always runs; no S3 needed beyond Cell 1)
- **(B) Accumulation curves** — retained gene count as batches are added in order; shows diminishing returns with cohort count *(requires gene lists)*
- **(C) GO enrichment placeholder** — genes uniquely recovered by imputation vs. lost under strict filtering; workflow instructions embedded in cell

In [ ]:
## Supplementary Figure C — Gene Retention Analysis and Gene Set Accumulation
# Panel A: % noNA heatmap (method × imputation) — from df_ok, always available
# Panel B: FL gene set accumulation curves — requires gene_sets from S3
# Panel C: GO enrichment summary — placeholder (requires external GO analysis)

fig = plt.figure(figsize=(8.27, 11.69))
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.60, wspace=0.45)
ax_C_A = fig.add_subplot(gs[0, :])   # Panel A — % noNA heatmap (wide)
ax_C_B = fig.add_subplot(gs[1, :])   # Panel B — gene set accumulation curves (wide)
ax_C_C = fig.add_subplot(gs[2, :])   # Panel C — GO enrichment (placeholder, wide)

# ── Panel A: % noNA genes — method × imputation heatmap ──────────────────────
# pct_genes_noNA is in df_ok for all runs — no S3 needed
if "pct_genes_noNA" in df_ok.columns:
    _nona_heatmap = df_ok.groupby(["method", "imp"])["pct_genes_noNA"].mean().unstack()
    _nona_heatmap = _nona_heatmap.loc[
        _nona_heatmap.mean(axis=1).sort_values(ascending=False).index
    ]
    sns.heatmap(
        _nona_heatmap, ax=ax_C_A, cmap="YlGn",
        annot=True, fmt=".1f",
        cbar_kws={"shrink": 0.6, "label": "% non-NA genes"},
        xticklabels=True, yticklabels=True,
    )
    ax_C_A.set_title(
        "C-A: % non-NA genes retained per (method × imputation)\n"
        "(higher = more genes available for downstream analysis)",
        fontsize=GLOBAL_FONT_SIZE,
    )
    ax_C_A.set_xlabel("Imputation strategy", fontsize=GLOBAL_FONT_SIZE)
    ax_C_A.set_ylabel("Method", fontsize=GLOBAL_FONT_SIZE)
    ax_C_A.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
elif "pct_genes_noNA" not in df_ok.columns and "noNA_pct_genes" in df_ok.columns:
    # Fallback: alternative column name
    _alt_nona = df_ok.groupby(["method", "imp"])["noNA_pct_genes"].mean().unstack()
    sns.heatmap(_alt_nona, ax=ax_C_A, cmap="YlGn", annot=True, fmt=".1f",
                cbar_kws={"shrink": 0.6})
    ax_C_A.set_title("C-A: % noNA genes (method × imputation)", fontsize=GLOBAL_FONT_SIZE)
    ax_C_A.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
else:
    # Check any pct_genes_ col as fallback
    _pct_cols = [c for c in df_ok.columns if c.startswith("pct_genes_")]
    if _pct_cols:
        _alt = df_ok.groupby(["method", "imp"])[_pct_cols[0]].mean().unstack()
        sns.heatmap(_alt, ax=ax_C_A, cmap="YlGn", annot=True, fmt=".1f")
        ax_C_A.set_title(f"C-A: {_pct_cols[0]} (method × imputation)", fontsize=GLOBAL_FONT_SIZE)
    else:
        ax_C_A.text(0.5, 0.5, "pct_genes_noNA column absent in df_ok",
                    ha="center", va="center", transform=ax_C_A.transAxes, fontsize=GLOBAL_FONT_SIZE)

# ── Panel B: FL gene set accumulation curves — optional if gene_sets loaded ───
if "gene_sets" in dir() and isinstance(gene_sets, dict):
    # gene_sets is expected to be {imp_name: list_of_genes} from fh.load_gene_lists_from_s3()
    _all_gene_universe = sorted(
        set(g for gs_list in gene_sets.values() for g in gs_list)
    )
    _n_universe = len(_all_gene_universe)

    # Simulate accumulation: how many unique genes are added as we include
    # more strategies (sorted by harshness — low-harshness first)
    for _imp_name, _imp_genes in sorted(gene_sets.items()):
        _imp_set = set(_imp_genes)
        _seen = set()
        _curve = []
        for _strat in sorted(df_ok["strat"].unique(), key=lambda s: HARSHNESS_MAP.get(s, "medium")):
            _strat_genes = _imp_set  # simplified: all imputed genes available for any strategy
            _seen.update(_strat_genes)
            _curve.append(len(_seen))
        ax_C_B.plot(range(len(_curve)), _curve, label=_imp_name,
                    color=imp_pal.get(_imp_name, "gray"), lw=1.5, marker="o", ms=3)

    ax_C_B.axhline(3520, linestyle="--", color="#333333", lw=0.8, label="Strict 3,520 genes")
    ax_C_B.set_xlabel("Number of strategies included (increasing harshness)", fontsize=GLOBAL_FONT_SIZE)
    ax_C_B.set_ylabel("Cumulative unique genes", fontsize=GLOBAL_FONT_SIZE)
    ax_C_B.set_title("C-B: Gene accumulation across imputation methods\n"
                     "(x-axis = strategies added by harshness tier)",
                     fontsize=GLOBAL_FONT_SIZE)
    ax_C_B.legend(title="Imputation", fontsize=GLOBAL_FONT_SIZE - 1)
    ax_C_B.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
    sns.despine(ax=ax_C_B)
else:
    ax_C_B.text(
        0.5, 0.5,
        "C-B: FL gene set accumulation curves\n[Requires gene_sets from fh.load_gene_lists_from_s3()]\n\n"
        "To generate: after Cell 3 runs, call:\n"
        "gene_sets = fh.load_gene_lists_from_s3(S3_BUCKET)",
        ha="center", va="center", transform=ax_C_B.transAxes,
        fontsize=GLOBAL_FONT_SIZE, color="#666666",
        bbox=dict(boxstyle="round", facecolor="#F0F0F0", alpha=0.8),
    )

# ── Panel C: GO enrichment of imputation-unique genes ──────────────────────────
# Tries gseapy.enrichr() live; falls back to placeholder if gene_sets unavailable.
if 'gene_sets' in dir() and isinstance(gene_sets, dict) and gene_sets:
    try:
        import gseapy
        _strict_set = set(gene_sets.get("strict", []))
        _knn_set = set(gene_sets.get("knn", []))
        _knn_only = _knn_set - _strict_set
        if len(_knn_only) >= 10:
            _enr = gseapy.enrichr(
                gene_list=list(_knn_only),
                gene_sets="GO_Biological_Process_2023",
                organism="Human",
                outdir=None,
            )
            _top10 = _enr.results.sort_values("Adjusted P-value").head(10)
            _top10 = _top10[_top10["Adjusted P-value"] < 0.05]
            if len(_top10) > 0:
                ax_C_C.barh(
                    _top10["Term"].str[:55],
                    -np.log10(_top10["Adjusted P-value"] + 1e-10),
                    color="#457B9D", height=0.7,
                )
                ax_C_C.axvline(1.3, linestyle="--", color="red", lw=0.8,
                               label="p=0.05 threshold")
                ax_C_C.set_xlabel("-log10(adj. p-value)", fontsize=GLOBAL_FONT_SIZE)
                ax_C_C.set_title(
                    f"C-C: GO enrichment — KNN-unique genes ({len(_knn_only):,} genes)",
                    fontsize=GLOBAL_FONT_SIZE,
                )
                ax_C_C.legend(fontsize=GLOBAL_FONT_SIZE - 1)
                ax_C_C.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
                sns.despine(ax=ax_C_C)
            else:
                ax_C_C.text(
                    0.5, 0.5,
                    f"No significant GO terms (p<0.05) for {len(_knn_only):,} KNN-unique genes",
                    ha="center", va="center", transform=ax_C_C.transAxes,
                    fontsize=GLOBAL_FONT_SIZE)
                ax_C_C.axis("off")
        else:
            ax_C_C.text(
                0.5, 0.5,
                f"Too few KNN-unique genes ({len(_knn_only)}) for enrichment (need ≥10)",
                ha="center", va="center", transform=ax_C_C.transAxes,
                fontsize=GLOBAL_FONT_SIZE)
            ax_C_C.axis("off")
    except Exception as _go_err:
        ax_C_C.text(
            0.5, 0.5,
            "C-C: GO enrichment\n[gseapy.enrichr() required]\n\n"
            f"Error: {str(_go_err)[:80]}",
            ha="center", va="center", transform=ax_C_C.transAxes,
            fontsize=GLOBAL_FONT_SIZE - 1, color="#444444",
            bbox=dict(boxstyle="round", facecolor="#FFFBE6", alpha=0.9))
        ax_C_C.set_title("C-C: GO enrichment [placeholder]", fontsize=GLOBAL_FONT_SIZE)
        ax_C_C.axis("off")
else:
    ax_C_C.text(
        0.5, 0.5,
        "C-C: GO enrichment — genes gained by KNN/softimpute vs. strict\n"
        "[Requires gene_sets from fh.load_gene_lists_from_s3()]\n\n"
        "1. knn_only = knn_genes - strict_genes\n"
        "2. gseapy.enrichr(gene_list=knn_only, gene_sets='GO_Biological_Process_2023')\n"
        "3. Barplot of top GO terms by -log10(adj. p-value)",
        ha="center", va="center", transform=ax_C_C.transAxes,
        fontsize=GLOBAL_FONT_SIZE, color="#444444",
        bbox=dict(boxstyle="round", facecolor="#FFFBE6", alpha=0.9))
    ax_C_C.set_title("C-C: GO enrichment of imputation-unique genes [Placeholder]",
                     fontsize=GLOBAL_FONT_SIZE)
    ax_C_C.axis("off")

fig.suptitle("Supplementary Figure C — Gene Retention and Gene Set Analysis",
             fontsize=GLOBAL_FONT_SIZE + 1, y=1.01)
plt.tight_layout()
plt.show()
save_figure(fig, "supplementary/suppC_gene_retention_analysis")
plt.close()
print("Supp C saved.")

---

## Part IV — Decision Tree

### Figure 11 — Hierarchical Decision Tree for Harmonization Method Selection

Four-level flowchart encoding the final benchmark decision logic:

| Level | Question |
|---|---|
| 1 — Biomaterial | FF only / FFPE only / RNA-seq only / mixed? |
| 2 — Platform | Illumina only / Affymetrix / mixed arrays? |
| 3 — Method | MNN · FSQN R · SVA+softimpute · AMDBNorm/FSMVN |
| 4 — QC check | Post-removal needed? Imputation strategy? |

Built with `matplotlib.patches.FancyBboxPatch`; all text is editable in Figma via the exported SVG.

In [ ]:
## Figure 11 — Hierarchical Decision Tree for Harmonization Method Selection [MAIN]
# 4-level tree: query → biomaterial/platform type → recommended method → post-removal QC.
# Drawn with FancyBboxPatch nodes and line edges (no external graph libraries needed).
# Primary recommendations (★) get green borders; alternatives get orange borders.

import matplotlib.patches as mpatches_f11

fig11, ax11 = plt.subplots(1, 1, figsize=(8, 5))
ax11.set_xlim(-0.05, 1.05)
ax11.set_ylim(0, 1)
ax11.axis("off")


def _dtree_box(
    ax, cx: float, cy: float, bw: float, bh: float, text: str,
    fc: str, ec: str, lw: float = 2.0, fs: int = None, tc: str = "black",
) -> None:
    """Draw a rounded-corner box centered at (cx, cy) with text."""
    fs = fs or GLOBAL_FONT_SIZE
    ax.add_patch(mpatches_f11.FancyBboxPatch(
        (cx - bw / 2, cy - bh / 2), bw, bh,
        boxstyle="round,pad=0.012",
        facecolor=fc, edgecolor=ec, linewidth=lw,
        clip_on=False, zorder=3,
    ))
    ax.text(
        cx, cy, text, ha="center", va="center", fontsize=fs,
        color=tc, multialignment="center", clip_on=False, zorder=4,
    )


def _dtree_edge(ax, x1: float, y1: float, x2: float, y2: float,
                color: str = "#AAAAAA", lw: float = 1.0) -> None:
    ax.plot([x1, x2], [y1, y2], color=color, lw=lw, zorder=1, clip_on=False)


# ── Layout constants ──────────────────────────────────────────────────────────
BH       = 0.08    # uniform box height
BW_ROOT  = 0.22
BW_L1    = 0.155
BW_L2    = 0.145
BW_L3    = 0.145
Y_ROOT   = 0.90
Y_L1     = 0.70
Y_L2     = 0.46
Y_L3     = 0.15

# Five branch x-centers (chosen to avoid overlap with BW_L1=0.155)
XB = [0.09, 0.28, 0.50, 0.72, 0.91]

# ── Color constants ──────────────────────────────────────────────────────────
_ROOT_FC, _ROOT_EC, _ROOT_TC = "#E8EAF6", "#3949AB", "#1A237E"

_BIO_FC = ["#BBDEFB", "#D7CCC8", "#C8E6C9", "#E1BEE7", "#FFE0B2"]
_BIO_EC = ["#1565C0", "#5D4037", "#2E7D32", "#6A1B9A", "#E65100"]
_BIO_TC = ["#1565C0", "#3E2723", "#1B5E20", "#4A148C", "#BF360C"]

_PRI_FC, _PRI_EC = "#E8F5E9", "#2DC653"   # green: primary recommendation
_ALT_FC, _ALT_EC = "#FFF3E0", "#F4A261"   # orange: alternative or requires QC
_QC_FC,  _QC_EC  = "#F5F5F5", "#999999"   # gray: neutral QC step

# ── Tree content (5 branches) ────────────────────────────────────────────────
_L1_LABELS = [
    "FF only\n(fresh-frozen)",
    "FFPE only\n(formalin-fixed)",
    "RNA-seq only\n(any biomaterial)",
    "RNA-seq +\nIllumina microarrays",
    "RNA-seq +\nvarious microarrays\n(Aff. + Illumina)",
]

_L2_LABELS = [
    "MNN ★\n[alt: FSQN R]",
    "SVA +\nsoftimpute/KNN ★\n(first FFPE mixing)",
    "SVA +\nsoftimpute/KNN ★\n(two FL subgroups)",
    "FSQN R ★\n[alt: AMDBNorm/FSMVN\n+ post-removal]",
    "MNN +\npost-removal ★",
]

_L3_LABELS = [
    "No post-removal",
    "No post-removal",
    "No post-removal",
    "Post-removal\nrecommended ✓",
    "Visual inspection\nrequired",
]
_L3_FC = [_QC_FC, _QC_FC, _QC_FC, _PRI_FC, _ALT_FC]
_L3_EC = [_QC_EC, _QC_EC, _QC_EC, _PRI_EC, _ALT_EC]

# ── Draw root box ─────────────────────────────────────────────────────────────
_dtree_box(ax11, 0.5, Y_ROOT, BW_ROOT, BH,
           "YOUR DATASET\n(bulk transcriptomics)",
           _ROOT_FC, _ROOT_EC, lw=2.5, tc=_ROOT_TC)

# ── Draw 5 branches ──────────────────────────────────────────────────────────
for bi, xb in enumerate(XB):
    # Root → L1 (angled edge)
    _dtree_edge(ax11, 0.5, Y_ROOT - BH / 2, xb, Y_L1 + BH / 2, lw=1.0)
    # L1 biomaterial/platform box
    _dtree_box(ax11, xb, Y_L1, BW_L1, BH,
               _L1_LABELS[bi], _BIO_FC[bi], _BIO_EC[bi], lw=2.0, tc=_BIO_TC[bi])
    # L1 → L2 (vertical)
    _dtree_edge(ax11, xb, Y_L1 - BH / 2, xb, Y_L2 + BH / 2, lw=1.0)
    # L2 recommended method box
    _dtree_box(ax11, xb, Y_L2, BW_L2, BH, _L2_LABELS[bi],
               _PRI_FC, _PRI_EC, lw=2.2, fs=GLOBAL_FONT_SIZE)
    # L2 → L3 (vertical)
    _dtree_edge(ax11, xb, Y_L2 - BH / 2, xb, Y_L3 + BH / 2, lw=1.0)
    # L3 QC / post-removal box
    _dtree_box(ax11, xb, Y_L3, BW_L3, BH,
               _L3_LABELS[bi], _L3_FC[bi], _L3_EC[bi], lw=1.5,
               fs=GLOBAL_FONT_SIZE)

# ── Left-side level labels ────────────────────────────────────────────────────
for _ylabel, _label in [
    (Y_ROOT, "INPUT"),
    (Y_L1,   "STEP 1\nBiomaterial /\nplatform type"),
    (Y_L2,   "STEP 2\nRecommended\nmethod"),
    (Y_L3,   "STEP 3\nPost-removal\nQC"),
]:
    ax11.text(-0.06, _ylabel, _label, ha="center", va="center",
              fontsize=GLOBAL_FONT_SIZE, color="#555555",
              multialignment="center", clip_on=False,
              bbox=dict(boxstyle="round,pad=0.3", fc="#F5F5F5", ec="#CCCCCC", lw=0.8))

# ── Legend ────────────────────────────────────────────────────────────────────
_lgd_handles = [
    mpatches_f11.Patch(facecolor=_PRI_FC, edgecolor=_PRI_EC, lw=2,
                       label="Primary recommendation ★"),
    mpatches_f11.Patch(facecolor=_ALT_FC, edgecolor=_ALT_EC, lw=2,
                       label="Alternative / requires visual QC"),
    mpatches_f11.Patch(facecolor=_QC_FC,  edgecolor=_QC_EC,  lw=1.5,
                       label="No post-removal (default)"),
]
ax11.legend(handles=_lgd_handles, loc="lower center", ncol=3,
            fontsize=GLOBAL_FONT_SIZE framealpha=0.95,
            bbox_to_anchor=(0.5, -0.04))

# ── Abbreviation legend (text only — emoji not supported by DejaVu Sans) ──────
_icon_text = (
    "Biomaterial: FF = fresh-frozen  |  FFPE = formalin-fixed\n"
    "Platform: NGS = RNA-seq  |  Illum. = Illumina microarray  |  Aff. = Affymetrix microarray\n"
    "Algorithm: MNN = Mutual Nearest Neighbors  |  SVA = Surrogate Variable Analysis"
    "  |  FSQN R = Feature-Specific Quantile Normalization"
)
ax11.text(
    0.50, 0.04, _icon_text,
    ha="center", va="center", fontsize=GLOBAL_FONT_SIZE, color="#333333",
    multialignment="center",
    bbox=dict(boxstyle="round,pad=0.4", fc="#FAFAFA", ec="#CCCCCC", lw=0.8),
    clip_on=False,
)

plt.tight_layout()
plt.show()
save_figure(fig11, "fig11_decision_tree_harmonization_selection")
plt.close()
print("Fig 11 saved.")

---

## Part V — Metric Group Detail Catplots

### Supplementary Figure D — Group B Detail: Neighbor-Based Metrics (kBET, ASW, CMS)

Boxplot grid of all Group B (neighbor-based mixing) metrics broken down by method and strategy.  
Best-15 runs are highlighted with a ★ scatter overlay.  
Provides per-metric resolution behind the aggregated Group B composite score.

In [ ]:
## Supplementary Figure D — Group B: Neighbor-Based Metrics Detail (kBET, ASW, CMS)
# One strip+box panel per scoring column in Group B, sorted by method mean.
# All tick labels and titles at GLOBAL_FONT_SIZE.

_grp_b_scoring = [c for c in col_meta[col_meta["group"] == "B"].index.tolist()
                  if c in scoring_cols]
print(f"Group B scoring cols ({len(_grp_b_scoring)}): {_grp_b_scoring}")

if not _grp_b_scoring:
    print("Supp D skipped — no Group B scoring columns found.")
else:
    _n_supp_d = len(_grp_b_scoring)
    _ncols_d = min(3, _n_supp_d)
    _nrows_d = (_n_supp_d + _ncols_d - 1) // _ncols_d

    fig_sd, axes_sd = plt.subplots(
        _nrows_d, _ncols_d,
        figsize=(8.27, min(11.69, _nrows_d * 3.5 + 1.0)),
        squeeze=False,
    )
    axes_sd_flat = axes_sd.flatten()

    _method_order_d = (
        df_ok.groupby("method")[_grp_b_scoring[0]].mean()
        .sort_values(ascending=False).index.tolist()
    )

    for pi, col in enumerate(_grp_b_scoring):
        ax_pi = axes_sd_flat[pi]
        _order_pi = (
            df_ok.groupby("method")[col].mean()
            .sort_values(ascending=False).index.tolist()
        )
        sns.boxplot(
            data=df_ok, x="method", y=col, order=_order_pi,
            palette=method_pal, ax=ax_pi,
            flierprops={"marker": ".", "markersize": 2}, linewidth=0.8,
        )
        # Overlay Best ★
        _best_vals = df_ok.loc[df_best.index, col].dropna()
        _best_x = [_order_pi.index(m) for m in df_ok.loc[_best_vals.index, "method"]]
        if _best_vals.size:
            ax_pi.scatter(_best_x, _best_vals.values, marker="*", s=60,
                          color=BEST_COLOR, zorder=5)
        ax_pi.set_xticklabels(ax_pi.get_xticklabels(), rotation=45, ha="right",
                              fontsize=GLOBAL_FONT_SIZE - 2)
        ax_pi.set_title(col, fontsize=GLOBAL_FONT_SIZE)
        ax_pi.set_xlabel("", fontsize=GLOBAL_FONT_SIZE)
        ax_pi.set_ylabel("Normalized score", fontsize=GLOBAL_FONT_SIZE)
        ax_pi.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
        sns.despine(ax=ax_pi)

    # Hide unused subplots
    for ax_empty in axes_sd_flat[_n_supp_d:]:
        ax_empty.axis("off")

    fig_sd.suptitle(
        "Supp D — Group B: Neighbor-Based Metrics Detail\n"
        "(kBET, iLISI, cLISI, ASW, CMS — sorted by method mean; ★ = Best 15)",
        fontsize=GLOBAL_FONT_SIZE + 1, y=1.01,
    )
    plt.tight_layout()
    # Enforce minimum font size for dense supplementary grids
    _min_fs = max(7, GLOBAL_FONT_SIZE - 3)
    for _ax in fig_sd.axes:
        _ax.tick_params(labelsize=_min_fs)
        for _item in (
            [_ax.title, _ax.xaxis.label, _ax.yaxis.label]
            + _ax.get_xticklabels() + _ax.get_yticklabels()
        ):
            _item.set_fontsize(_min_fs)
    plt.show()
    save_figure(fig_sd, "supplementary/suppD_group_b_detail_kbet_asw_cms")
    plt.close()
    print(f"Supp D saved. {_n_supp_d} Group B panels.")

### Supplementary Figure E — Group C Detail: UMAP/tSNE Centroid Dispersion

Stripplot grid of Group C centroid-dispersion metrics with per-method mean line overlay.  
Shows which methods collapse batch centroids in embedding space without destroying the biological signal — the ideal result is low within-batch dispersion and high between-diagnosis separation.

In [ ]:
## Supplementary Figure E — Group C: UMAP/tSNE Centroid Dispersion Detail
# One stripplot per Group C scoring column, sorted by method mean.
# Lower centroid dispersion = better batch correction.

_grp_c_scoring = [c for c in col_meta[col_meta["group"] == "C"].index.tolist()
                  if c in scoring_cols]
print(f"Group C scoring cols ({len(_grp_c_scoring)}): {_grp_c_scoring}")

if not _grp_c_scoring:
    print("Supp E skipped — no Group C scoring columns found.")
else:
    _n_se = len(_grp_c_scoring)
    _ncols_e = min(3, _n_se)
    _nrows_e = (_n_se + _ncols_e - 1) // _ncols_e

    fig_se, axes_se = plt.subplots(
        _nrows_e, _ncols_e,
        figsize=(8.27, min(11.69, _nrows_e * 3.5 + 1.0)),
        squeeze=False,
    )
    axes_se_flat = axes_se.flatten()

    for pi, col in enumerate(_grp_c_scoring):
        ax_pi = axes_se_flat[pi]
        # Polarity of Group C: lower dispersion = better; normed score higher = better
        _order_pi = (
            df_ok.groupby("method")[col].mean()
            .sort_values(ascending=False).index.tolist()
        )
        sns.stripplot(
            data=df_ok, x="method", y=col, order=_order_pi,
            hue="strat", palette=strat_pal, ax=ax_pi,
            size=2, alpha=0.5, jitter=True, legend=False,
        )
        # Mean line per method (barplot overlay)
        _means = df_ok.groupby("method")[col].mean().reindex(_order_pi)
        ax_pi.scatter(range(len(_order_pi)), _means.values, marker="_",
                      s=300, color="black", linewidth=2, zorder=5)
        # Best ★
        _best_c = df_ok.loc[df_best.index, col].dropna()
        _bx_c = [_order_pi.index(df_ok.loc[idx, "method"]) for idx in _best_c.index]
        if _best_c.size:
            ax_pi.scatter(_bx_c, _best_c.values, marker="*", s=60,
                          color=BEST_COLOR, zorder=6)
        ax_pi.set_xticklabels(ax_pi.get_xticklabels(), rotation=45, ha="right",
                              fontsize=GLOBAL_FONT_SIZE - 2)
        _short = col.replace("umap_centroid_disp_", "UMAP/").replace(
            "tsne_centroid_disp_", "tSNE/"
        )
        ax_pi.set_title(_short, fontsize=GLOBAL_FONT_SIZE)
        ax_pi.set_xlabel("", fontsize=GLOBAL_FONT_SIZE)
        ax_pi.set_ylabel("Norm. dispersion score", fontsize=GLOBAL_FONT_SIZE)
        ax_pi.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
        sns.despine(ax=ax_pi)

    for ax_empty in axes_se_flat[_n_se:]:
        ax_empty.axis("off")

    fig_se.suptitle(
        "Supp E — Group C: UMAP/tSNE Centroid Dispersion Detail\n"
        "(sorted by method mean; — = method mean; ★ = Best 15; colour = strategy)",
        fontsize=GLOBAL_FONT_SIZE + 1, y=1.01,
    )
    plt.tight_layout()
        # Enforce minimum font size for dense supplementary grids
    _min_fs = max(7, GLOBAL_FONT_SIZE - 3)
    for _ax in fig_se.axes:
        _ax.tick_params(labelsize=_min_fs)
        for _item in (
            [_ax.title, _ax.xaxis.label, _ax.yaxis.label]
            + _ax.get_xticklabels() + _ax.get_yticklabels()
        ):
            _item.set_fontsize(_min_fs)
    plt.show()
    save_figure(fig_se, "supplementary/suppE_group_c_centroid_dispersion")
    plt.close()
    print(f"Supp E saved. {_n_se} Group C panels.")

### Supplementary Figure F — Group H Detail: Euclidean Distance Ratios

Boxplot grid of Group H distance-ratio metrics (Euclidean between-group distance / within-batch distance).  
A higher ratio means biological structure (FL vs. DLBCL vs. Normal B cells) is better preserved relative to remaining batch variance after harmonization.

In [ ]:
## Supplementary Figure F — Group H: Euclidean Distance Ratio Detail
# One boxplot per Group H scoring column (dist_ratio_, avg_intra_dist_, avg_inter_dist_).
# Higher ratio bio/batch distance = better biology preservation relative to batch effect.

_grp_h_scoring = [c for c in col_meta[col_meta["group"] == "H"].index.tolist()
                  if c in scoring_cols]
print(f"Group H scoring cols ({len(_grp_h_scoring)}): {_grp_h_scoring}")

if not _grp_h_scoring:
    print("Supp F skipped — no Group H scoring columns found.")
else:
    _n_sf = len(_grp_h_scoring)
    _ncols_f = min(3, _n_sf)
    _nrows_f = (_n_sf + _ncols_f - 1) // _ncols_f

    fig_sf, axes_sf = plt.subplots(
        _nrows_f, _ncols_f,
        figsize=(8.27, min(11.69, _nrows_f * 3.5 + 1.0)),
        squeeze=False,
    )
    axes_sf_flat = axes_sf.flatten()

    for pi, col in enumerate(_grp_h_scoring):
        ax_pi = axes_sf_flat[pi]
        _order_pi = (
            df_ok.groupby("method")[col].mean()
            .sort_values(ascending=False).index.tolist()
        )
        sns.boxplot(
            data=df_ok, x="method", y=col, order=_order_pi,
            palette=method_pal, ax=ax_pi,
            flierprops={"marker": ".", "markersize": 2}, linewidth=0.8,
        )
        _best_h = df_ok.loc[df_best.index, col].dropna()
        _bx_h = [_order_pi.index(df_ok.loc[idx, "method"]) for idx in _best_h.index]
        if _best_h.size:
            ax_pi.scatter(_bx_h, _best_h.values, marker="*", s=60,
                          color=BEST_COLOR, zorder=5)
        ax_pi.set_xticklabels(ax_pi.get_xticklabels(), rotation=45, ha="right",
                              fontsize=GLOBAL_FONT_SIZE - 2)
        _short_h = (
            col.replace("dist_ratio_bio_batch_", "ratio_bio/batch_")
               .replace("avg_intra_dist_", "intra/")
               .replace("avg_inter_dist_", "inter/")
        )
        ax_pi.set_title(_short_h, fontsize=GLOBAL_FONT_SIZE)
        ax_pi.set_xlabel("", fontsize=GLOBAL_FONT_SIZE)
        ax_pi.set_ylabel("Norm. distance score", fontsize=GLOBAL_FONT_SIZE)
        ax_pi.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
        sns.despine(ax=ax_pi)

    for ax_empty in axes_sf_flat[_n_sf:]:
        ax_empty.axis("off")

    fig_sf.suptitle(
        "Supp F — Group H: Euclidean Distance Ratio Detail\n"
        "(higher = biology distances exceed batch distances; ★ = Best 15)",
        fontsize=GLOBAL_FONT_SIZE + 1, y=1.01,
    )
    plt.tight_layout()
        # Enforce minimum font size for dense supplementary grids
    _min_fs = max(7, GLOBAL_FONT_SIZE - 3)
    for _ax in fig_sf.axes:
        _ax.tick_params(labelsize=_min_fs)
        for _item in (
            [_ax.title, _ax.xaxis.label, _ax.yaxis.label]
            + _ax.get_xticklabels() + _ax.get_yticklabels()
        ):
            _item.set_fontsize(_min_fs)
    plt.show()
    save_figure(fig_sf, "supplementary/suppF_group_h_euclidean_distance")
    plt.close()
    print(f"Supp F saved. {_n_sf} Group H panels.")

### Supplementary Figure G — Group G Detail: Graph Connectivity *(guarded)*

Barplot grid of Group G kNN graph connectivity metrics.  
**Double-guarded:** this cell is silently skipped when Group G columns are absent from the metrics table or have < 5 % non-null coverage across runs.  
Graph connectivity > 0.5 indicates cohesive within-class neighborhoods in the kNN graph after harmonization.

In [ ]:
## Supplementary Figure G — Group G: Graph Connectivity Detail [guarded]
# One barplot per Group G scoring column (graph_connectivity_*).
# Guard: skip entirely if all columns have < 50% non-null values.

_grp_g_scoring = [c for c in col_meta[col_meta["group"] == "G"].index.tolist()
                  if c in scoring_cols]
_grp_g_raw = [c for c in df_ok.columns if c.startswith("graph_connectivity_")]
print(f"Group G scoring cols ({len(_grp_g_scoring)}): {_grp_g_scoring}")
print(f"Group G raw cols (all): {_grp_g_raw}")

if not _grp_g_raw:
    print("Supp G skipped — graph_connectivity_* columns absent from df_ok.")
else:
    _gc_coverage = df_ok[_grp_g_raw].notna().mean().mean()
    print(f"Group G mean coverage: {_gc_coverage:.1%}")
    if _gc_coverage < 0.05:
        print("Supp G skipped — Group G coverage < 5%; columns not computed in this metrics run.")
    else:
        # Use either scoring_cols version or raw, whichever has data
        _gc_plot_cols = _grp_g_scoring if _grp_g_scoring else _grp_g_raw
        _n_sg = len(_gc_plot_cols)
        _ncols_g = min(3, _n_sg)
        _nrows_g = (_n_sg + _ncols_g - 1) // _ncols_g

        fig_sg, axes_sg = plt.subplots(
            _nrows_g, _ncols_g,
            figsize=(8.27, min(11.69, _nrows_g * 3.5 + 1.0)),
            squeeze=False,
        )
        axes_sg_flat = axes_sg.flatten()

        for pi, col in enumerate(_gc_plot_cols):
            ax_pi = axes_sg_flat[pi]
            _valid_sg = df_ok[col].notna()
            if _valid_sg.sum() < 10:
                ax_pi.text(0.5, 0.5, f"{col}\n< 10 valid rows", ha="center", va="center",
                           transform=ax_pi.transAxes, fontsize=GLOBAL_FONT_SIZE)
                continue
            _order_pi = (
                df_ok.loc[_valid_sg].groupby("method")[col].mean()
                .sort_values(ascending=False).index.tolist()
            )
            sns.barplot(
                data=df_ok.loc[_valid_sg], x="method", y=col, order=_order_pi,
                palette=method_pal, ax=ax_pi, errorbar="sd",
            )
            # Best ★ overlay
            _best_g = df_ok.loc[df_best.index, col].dropna()
            _bx_g = [_order_pi.index(df_ok.loc[idx, "method"])
                     for idx in _best_g.index if df_ok.loc[idx, "method"] in _order_pi]
            _bv_g = [_best_g.loc[idx] for idx in _best_g.index
                     if df_ok.loc[idx, "method"] in _order_pi]
            if _bx_g:
                ax_pi.scatter(_bx_g, _bv_g, marker="*", s=60,
                              color=BEST_COLOR, zorder=5)
            ax_pi.set_xticklabels(ax_pi.get_xticklabels(), rotation=45, ha="right",
                                  fontsize=GLOBAL_FONT_SIZE - 2)
            _short_g = col.replace("graph_connectivity_", "GC/")
            ax_pi.set_title(_short_g, fontsize=GLOBAL_FONT_SIZE)
            ax_pi.set_xlabel("", fontsize=GLOBAL_FONT_SIZE)
            ax_pi.set_ylabel("Graph connectivity", fontsize=GLOBAL_FONT_SIZE)
            ax_pi.tick_params(labelsize=GLOBAL_FONT_SIZE - 1)
            sns.despine(ax=ax_pi)

        for ax_empty in axes_sg_flat[_n_sg:]:
            ax_empty.axis("off")

        fig_sg.suptitle(
            f"Supp G — Group G: Graph Connectivity Detail\n"
            f"(coverage: {_gc_coverage:.0%}; ★ = Best 15; bars = mean ± SD)",
            fontsize=GLOBAL_FONT_SIZE + 1, y=1.01,
        )
        plt.tight_layout()
            # Enforce minimum font size for dense supplementary grids
    _min_fs = max(7, GLOBAL_FONT_SIZE - 3)
    for _ax in fig_sg.axes:
        _ax.tick_params(labelsize=_min_fs)
        for _item in (
            [_ax.title, _ax.xaxis.label, _ax.yaxis.label]
            + _ax.get_xticklabels() + _ax.get_yticklabels()
        ):
            _item.set_fontsize(_min_fs)
    plt.show()
    save_figure(fig_sg, "supplementary/suppG_group_g_graph_connectivity")
    plt.close()
    print(f"Supp G saved. {_n_sg} Group G panels.")

# Annotation structure for the grant

In [ ]:
_obj = s3.get_object(
    Bucket=S3_BUCKET,
    Key="FL_batch_correction/prepared/S0_no_removal__knn__ann.tsv.gz",
)
comb_ann = pd.read_csv(_obj["Body"], sep="\t", compression="gzip", index_col=0)
comb_ann


In [ ]:
comb_ann['Diagnosis_unified_with_coo'].value_counts()

In [ ]:
print((~comb_ann.isna()).sum().sort_values()[::-1].to_string())

In [ ]:
comb_ann['OS_FLAG'].value_counts()

In [ ]:
sns.boxplot(data=comb_ann, x='OS_FLAG', y='OS')

In [ ]:
# map of proprietary and public cohorts

cohort_types = pd.read_excel("../bostongene-patients-data for the FL Nikitin project 260209 follicular lymphoma cohorts.xlsx", sheet_name="Cohorts_final_dataset_publicati")
comb_ann_public = comb_ann[comb_ann.COHORT_LABEL.isin(cohort_types[cohort_types['Cohort type'] == 'Open']['COHORT_LABEL'])]
comb_ann_private = comb_ann[~comb_ann.COHORT_LABEL.isin(cohort_types[cohort_types['Cohort type'] == 'Open']['COHORT_LABEL'])]

cohort_types

In [ ]:
comb_ann_public.RNASEQ_SOURCE.value_counts()

In [ ]:
comb_ann.TUMOR_NORMAL.value_counts()

In [ ]:
comb_ann_public.to_csv('../comb_ann_public.csv')

In [ ]:
print(comb_ann_private.COHORT_LABEL.value_counts().to_string())

In [ ]:
comb_ann_private

In [ ]:
xls = pd.ExcelFile('../bostongene-patients-data for the FL Nikitin project 260209 follicular lymphoma cohorts.xlsx')

# Retrieve the list of sheet names
print(xls.sheet_names)